# **Explainable Graph Neural Networks for Graph Matching in Building Information Modelling**

### Project Topic: **Experiments with Graph Neural Networks in the Construction Industry (Building Information Modelling)**
  
---

## **Group Members**
- **Reza Almassi**  
- **Ronald Omoding**  
- **Tewodros Abere Muche**

---

## **Project Description**

This project aims to improve the **performance and interpretability** of a graph‑matching model for **hierarchical scene graphs** (rooms + wall surfaces). This project is inspired by two key papers:

- [**Ndulue et al. (2026)**](https://arxiv.org/abs/2604.27821) — *Learning-Based Hierarchical Scene Graph Matching for Robot Localization Leveraging Prior Maps*  

- [**Shaheer et al. (2023)**](https://arxiv.org/abs/2303.02076)   — *Graph-based Global Robot Localization Informing Situational Graphs with Architectural Graphs*

Our goal is to:

- evaluate **alternative GNN architectures** for graph matching,  
- compare their performance on a **preprocessed [MSD dataset](https://arxiv.org/html/2407.10121v1)**,  
- and apply **explainability techniques** to identify which nodes and edges contribute most to **incorrect predictions**.




## Setup

### Install required packages

In [ ]:
!pip install torch-geometric -q
!pip install pygmtools -q
!pip install shapely -q
!pip install optuna -q
!pip install networkx matplotlib pandas tqdm seaborn -q

### Import required libraries

In [ ]:
import os
import pickle
import random
import copy
import time
import warnings
from pathlib import Path
import urllib.request
import zipfile
from typing import List, Tuple, Dict, Optional, Any
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import networkx as nx
import seaborn as sns
from sklearn.metrics import classification_report

import optuna
from optuna.trial import TrialState
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, SAGEConv, GINConv, GATConv, GATv2Conv, TransformerConv
from torch_geometric.utils import to_networkx

from shapely.geometry import Polygon
from shapely.affinity import translate
from tqdm import tqdm

import pygmtools
pygmtools.BACKEND = 'pytorch'

warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Create results folder
os.makedirs('/content/results', exist_ok=True)

### Download Dataset from Dropbox

In [ ]:
DROPBOX_URL = "https://www.dropbox.com/scl/fi/kdv2gf1h76sdtfjbre3mo/msd_dataset.zip?rlkey=b9mc3bsgfjtxbq9s3yl07sn3o&st=j7ns3v3w&dl=1"

def download_dataset(url, output_path="msd_dataset.zip"):
    """Download dataset from Dropbox"""
    print(f"Downloading dataset from {url}...")
    urllib.request.urlretrieve(url, output_path)
    print(f"Downloaded to {output_path}")
    return output_path

def extract_zip(zip_path, extract_to="."):
    """Extract zip file"""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Extracted to {extract_to}")

zip_path = download_dataset(DROPBOX_URL)
extract_zip(zip_path, "/content/msd_data")

# Delete ZIP after extraction
os.remove(zip_path)
print(f"Deleted zip file: {zip_path}")

### Download Pretrained Models from Dropbox

In [ ]:
MODELS_DROPBOX_URL = "https://www.dropbox.com/scl/fi/ba31vjdc1xaoiuit6w48m/pretrained_models.zip?rlkey=olyyvhnd5ujm04px9353e7drp&st=kbi0lssm&dl=1"

def download_pretrained_models(url, output_path="pretrained_models.zip"):
    """Download pretrained models from Dropbox"""
    print(f"Downloading pretrained models from {url}...")
    urllib.request.urlretrieve(url, output_path)
    print(f"Downloaded to {output_path}")
    return output_path

def extract_zip(zip_path, extract_to="."):
    """Extract zip file"""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Extracted to {extract_to}")

zip_path = download_pretrained_models(MODELS_DROPBOX_URL)
extract_zip(zip_path, "/content/pretrained_models")

# Delete ZIP after extraction
os.remove(zip_path)
print(f"Deleted zip file: {zip_path}")

## Utility Functions

### Dataset Utility Functions

In [ ]:
node_type_mapping = {"room": [1, 0], "ws": [0, 1]}

def nx_to_pyg_data_preserve_order(graph: nx.DiGraph) -> Data:
    """
    Convert a NetworkX DiGraph to a PyTorch Geometric Data object,
    preserving node insertion order.
    """
    node_ids = list(graph.nodes())
    id_map = {nid: i for i, nid in enumerate(node_ids)}

    x = torch.stack([
        torch.tensor(
            node_type_mapping[graph.nodes[n]['type']] +
            graph.nodes[n]['center'] +
            graph.nodes[n]['normal'] +
            [graph.nodes[n].get('length', -1)],
            dtype=torch.float32
        )
        for n in node_ids
    ])

    edge_index = torch.tensor(
        [[id_map[u], id_map[v]] for u, v in graph.edges()],
        dtype=torch.long
    ).t().contiguous() if graph.edges else torch.empty((2, 0), dtype=torch.long)

    data = Data(x=x, edge_index=edge_index)
    data.name = graph.graph.get('name', '')
    data.node_names = node_ids
    data.permutation = torch.arange(len(node_ids), dtype=torch.long)
    return data

def pyg_data_to_nx_digraph(data: Data, graph_list: List[nx.DiGraph]) -> nx.DiGraph:
    """Convert PyG Data back to NetworkX DiGraph"""
    matching_graph = next((g for g in graph_list if g.graph.get('name') == data.name), None)
    if matching_graph is None:
        raise ValueError(f"No graph with name {data.name} found.")

    orig_names = data.node_names
    perm = data.permutation.tolist()
    node_ids = [orig_names[idx] for idx in perm]

    G = nx.DiGraph()
    for node_id in node_ids:
        if node_id in matching_graph.nodes:
            G.add_node(node_id, **matching_graph.nodes[node_id])

    for u_idx, v_idx in data.edge_index.t().tolist():
        u = node_ids[u_idx]
        v = node_ids[v_idx]
        if matching_graph.has_edge(u, v):
            G.add_edge(u, v, **matching_graph.edges[u, v])

    G.graph['name'] = data.name
    return G

def deserialize_graph_matching_dataset(path: str, filename: str = "train_dataset.pkl") -> List[Tuple[Data, Data, torch.Tensor]]:
    """Deserialize dataset of (Data1, Data2, PermutationMatrix) tuples"""
    full_path = os.path.join(path, filename)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"File not found: {full_path}")
    with open(full_path, 'rb') as f:
        pairs = pickle.load(f)
    print(f"Loaded {len(pairs)} pairs from {full_path}")
    return pairs

def serialize_graph_matching_dataset(pairs: List[Tuple[Data, Data, torch.Tensor]], path: str, filename: str = "dataset.pkl"):
    """Serialize dataset to file"""
    os.makedirs(path, exist_ok=True)
    full_path = os.path.join(path, filename)
    with open(full_path, 'wb') as f:
        pickle.dump(pairs, f)
    print(f"Serialized {len(pairs)} pairs to {full_path}")

def compute_mean_std(pairs: List[Tuple[Data, Data, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute per-feature mean and std from training set"""
    x_list = []
    for data1, data2, _ in pairs:
        x_list.append(data1.x)
        x_list.append(data2.x)
    x_all = torch.cat(x_list, dim=0)
    mean = x_all.mean(dim=0)
    std = x_all.std(dim=0)
    return mean, std

def normalize_data_pairs(pairs: List[Tuple[Data, Data, torch.Tensor]], mean: torch.Tensor, std: torch.Tensor) -> List[Tuple[Data, Data, torch.Tensor]]:
    """Normalize features in Data objects."""
    normalized_pairs = []
    for data1, data2, P in pairs:
        data1_clone = Data(x=(data1.x - mean) / (std + 1e-8), edge_index=data1.edge_index)
        data2_clone = Data(x=(data2.x - mean) / (std + 1e-8), edge_index=data2.edge_index)
        # Preserve metadata
        if hasattr(data1, 'name'):
            data1_clone.name = data1.name
        if hasattr(data2, 'name'):
            data2_clone.name = data2.name
        normalized_pairs.append((data1_clone, data2_clone, P))
    return normalized_pairs

def collate_pyg_matching(batch):
    """Custom collate function for graph matching batches"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data1_list, data2_list, perm_list = zip(*batch)

    # Move to device
    data1_list = [d.to(device) for d in data1_list]
    data2_list = [d.to(device) for d in data2_list]

    # Create batches
    batch1 = Batch.from_data_list(data1_list)
    batch2 = Batch.from_data_list(data2_list)

    return batch1, batch2, perm_list

class GraphMatchingDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

### Visualization Utility Functions

In [ ]:
def plot_a_graph(graphs_list, ax=None, viz_rooms=True, viz_ws=True,
                 viz_room_connection=True, viz_normals=False,
                 viz_room_normals=False, viz_walls=True, title=None):
    """
    Visualizes geometries, wall segments, and graph edges for multiple apartments in 2D.
    """
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(12, 10))

    legend_added = set()
    normal_added = False

    for graphs in graphs_list:
        # Visualize room polygons
        if viz_rooms:
            room_nodes = [n for n, d in graphs.nodes(data=True) if d['type'] == 'room']
            for idx, room_node in enumerate(room_nodes):
                room_data = graphs.nodes[room_node]
                if 'polygon' in room_data:
                    room_polygon = Polygon(room_data['polygon'])
                    x, y = room_polygon.exterior.xy
                    if "Room polygon" not in legend_added:
                        ax.plot(x, y, color='black', alpha=0.3, linewidth=2, label='Room polygon')
                        legend_added.add("Room polygon")
                    else:
                        ax.plot(x, y, color='black', alpha=0.3, linewidth=2)
                # Draw room centroids
                if "Room centroid" not in legend_added:
                    ax.scatter(room_data['center'][0], room_data['center'][1], color='blue', s=100,
                              edgecolors='darkblue', linewidth=2, zorder=3, label='Room centroid')
                    legend_added.add("Room centroid")
                else:
                    ax.scatter(room_data['center'][0], room_data['center'][1], color='blue', s=100,
                              edgecolors='darkblue', linewidth=2, zorder=3)

        # Visualize WS nodes
        if viz_ws:
            ws_nodes = [n for n, d in graphs.nodes(data=True) if d['type'] == 'ws']
            for idx, wn in enumerate(ws_nodes):
                ws_data = graphs.nodes[wn]
                if "WS" not in legend_added:
                    ax.scatter(ws_data['center'][0], ws_data['center'][1], color='red', s=40,
                              edgecolors='darkred', linewidth=1.5, zorder=3, label='Wall segment')
                    legend_added.add("WS")
                else:
                    ax.scatter(ws_data['center'][0], ws_data['center'][1], color='red', s=40,
                              edgecolors='darkred', linewidth=1.5, zorder=3)

                if viz_room_normals:
                    if not normal_added:
                        ax.arrow(ws_data['center'][0], ws_data['center'][1],
                                ws_data['normal'][0], ws_data['normal'][1],
                                head_width=0.15, head_length=0.15, fc='green', ec='green',
                                alpha=0.7, label='Normal')
                        normal_added = True
                    else:
                        ax.arrow(ws_data['center'][0], ws_data['center'][1],
                                ws_data['normal'][0], ws_data['normal'][1],
                                head_width=0.15, head_length=0.15, fc='green', ec='green', alpha=0.7)

                if 'limits' in ws_data:
                    limit_1, limit_2 = ws_data['limits']
                    ax.plot([limit_1[0], limit_2[0]], [limit_1[1], limit_2[1]],
                            color='black', linewidth=2.0, alpha=0.8)

            # Draw WS edges
            ws_edges = [(u, v) for u, v, d in graphs.edges(data=True)
                       if 'ws_same_room' in d.get('type', '') or 'ws_belongs_room' in d.get('type', '')]
            for idx, edge in enumerate(ws_edges):
                start_node = graphs.nodes[edge[0]]
                end_node = graphs.nodes[edge[1]]
                if "WS edge" not in legend_added:
                    ax.plot([start_node['center'][0], end_node['center'][0]],
                           [start_node['center'][1], end_node['center'][1]],
                           color='gray', linestyle='--', alpha=0.6, linewidth=1.5, label='WS connection')
                    legend_added.add("WS edge")
                else:
                    ax.plot([start_node['center'][0], end_node['center'][0]],
                           [start_node['center'][1], end_node['center'][1]],
                           color='gray', linestyle='--', alpha=0.6, linewidth=1.5)

        # Visualize room connections
        if viz_room_connection:
            connection_edges = [(u, v) for u, v, d in graphs.edges(data=True)
                               if 'connected' in d.get('type', '')]
            for idx, edge in enumerate(connection_edges):
                start_node = graphs.nodes[edge[0]]
                end_node = graphs.nodes[edge[1]]
                if "Room connection" not in legend_added:
                    ax.plot([start_node['center'][0], end_node['center'][0]],
                           [start_node['center'][1], end_node['center'][1]],
                           color='blue', linestyle='-', alpha=0.5, linewidth=1, label='Room adjacency')
                    legend_added.add("Room connection")
                else:
                    ax.plot([start_node['center'][0], end_node['center'][0]],
                           [start_node['center'][1], end_node['center'][1]],
                           color='blue', linestyle='-', alpha=0.5, linewidth=1)

    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    return ax


def plot_two_graphs_with_matching(graphs_list, gt_perm, original_graphs,
                                   pred_perm=None, noise_graphs=None,
                                   viz_rooms=True, viz_ws=True,
                                   viz_room_connection=True,
                                   viz_normals=False, viz_room_normals=False,
                                   match_display="all", title=None, save_path=None):
    """
    Visualizes two graphs side-by-side with matching lines.
    Green lines = correct matches, Red lines = wrong matches.
    """
    assert match_display in {"all", "correct", "wrong"}, "match_display must be 'all', 'correct', or 'wrong'"
    assert len(graphs_list) == 2, "graphs_list must contain exactly two graphs."

    if noise_graphs is None:
        noise_graphs = original_graphs

    # Extract tensors and original node order
    g1tensor, g2tensor = copy.deepcopy(graphs_list[0]), copy.deepcopy(graphs_list[1])
    node_names1 = list(g1tensor.node_names)
    orig_names2 = list(g2tensor.node_names)
    perm = g2tensor.permutation.tolist()
    node_names2 = [orig_names2[p] for p in perm]

    # Convert to NetworkX
    g1 = pyg_data_to_nx_digraph(g1tensor, original_graphs)
    g2_original = pyg_data_to_nx_digraph(g2tensor, noise_graphs)
    g2 = g2_original.copy()

    # Translate g2 for side-by-side plot
    max_x_g1 = max(data['center'][0] for _, data in g1.nodes(data=True))
    min_x_g2 = min(data['center'][0] for _, data in g2.nodes(data=True))
    translation_x = (max_x_g1 - min_x_g2) + 10.0
    for _, data in g2.nodes(data=True):
        data['center'][0] += translation_x
        if 'polygon' in data:
            poly = data['polygon']
            if isinstance(poly, Polygon):
                data['polygon'] = translate(poly, xoff=translation_x)
            else:
                data['polygon'] = Polygon([(x + translation_x, y) for x, y in poly])
        if 'limits' in data:
            data['limits'] = [[x + translation_x, y] for x, y in data['limits']]

    fig, ax = plt.subplots(figsize=(18, 10))
    legend_added = set()

    def plot_graph(g, is_g1):
        color_room = 'lightblue' if is_g1 else 'navajowhite'
        color_ws = 'red' if is_g1 else 'purple'
        prefix = "A-graph" if is_g1 else "S-graph"

        # Plot rooms
        for n, d in g.nodes(data=True):
            if d['type'] == 'room' and 'polygon' in d:
                poly = Polygon(d['polygon']) if not isinstance(d['polygon'], Polygon) else d['polygon']
                x, y = poly.exterior.xy
                ax.fill(x, y, color=color_room, alpha=0.3,
                       label=f"{prefix} room" if f"room-{prefix}" not in legend_added else "")
                ax.scatter(d['center'][0], d['center'][1], color='blue', s=120,
                          edgecolors='darkblue', linewidth=2, zorder=3,
                          label=f"{prefix} centroid" if f"centroid-{prefix}" not in legend_added else "")
                legend_added.update({f"room-{prefix}", f"centroid-{prefix}"})

        # Plot WS nodes
        for n, d in g.nodes(data=True):
            if d['type'] == 'ws':
                ax.scatter(d['center'][0], d['center'][1], color=color_ws, s=50,
                          edgecolors='darkred' if is_g1 else 'purple', linewidth=1.5, zorder=3,
                          label=f"{prefix} WS" if f"ws-{prefix}" not in legend_added else "")
                legend_added.add(f"ws-{prefix}")
                if 'limits' in d:
                    limit1, limit2 = d['limits']
                    ax.plot([limit1[0], limit2[0]], [limit1[1], limit2[1]],
                           color='black', linewidth=2.0, alpha=0.8,
                           label=f"{prefix} wall" if f"wall-{prefix}" not in legend_added else "")
                    legend_added.add(f"wall-{prefix}")

    plot_graph(g1, is_g1=True)
    plot_graph(g2, is_g1=False)

    # Plot matching lines
    if pred_perm is not None:
        for i in range(pred_perm.shape[0]):
            if gt_perm[i].sum().item() == 0:
                continue

            row = pred_perm[i]
            if row.sum().item() == 0:
                j_gt = gt_perm[i].argmax().item()
                id1 = node_names1[i]
                if id1 not in g1.nodes:
                    continue
                if j_gt < len(node_names2):
                    id2 = node_names2[j_gt]
                else:
                    continue
                if id2 not in g2.nodes:
                    continue
                pt1 = g1.nodes[id1]['center']
                pt2 = g2.nodes[id2]['center']
                if match_display in {"correct"}:
                    continue
                ax.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]],
                       color='orange', linestyle='--', alpha=0.6, linewidth=2,
                       label='Missing match' if 'missing' not in legend_added else "")
                legend_added.add('missing')
                continue

            j = row.argmax().item()
            id1 = node_names1[i]
            if id1 not in g1.nodes:
                continue
            if j < len(node_names2):
                id2 = node_names2[j]
            else:
                continue
            if id2 not in g2.nodes:
                continue

            pt1 = g1.nodes[id1]['center']
            pt2 = g2.nodes[id2]['center']
            is_correct = (j < gt_perm.shape[1] and gt_perm[i, j] == 1)

            if match_display == "correct" and not is_correct:
                continue
            if match_display == "wrong" and is_correct:
                continue

            color = 'green' if is_correct else 'red'
            label = None
            if color == 'green' and 'correct' not in legend_added:
                label = 'Correct match'
                legend_added.add('correct')
            elif color == 'red' and 'wrong' not in legend_added:
                label = 'Wrong match'
                legend_added.add('wrong')

            ax.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]],
                   color=color, linestyle='-', alpha=0.7, linewidth=2.5, label=label)

    ax.set_title(title if title else "Graph Matching Results", fontsize=16, fontweight='bold')
    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9, ncol=2)
    ax.grid(True, alpha=0.3)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved visualization to {save_path}")

    plt.tight_layout()
    return fig, ax

### Training Utility Functions

In [ ]:
def compute_metrics(S_pred, P_gt, threshold=0.5):
    """
    Paper Section IV-A: Binary classification over all N2 × N1 candidate node pairs.
    """
    N1, N2 = P_gt.shape

    S_real = S_pred[:, :N2]
    hard_assign = pygmtools.hungarian(S_real)

    tp = 0  # Correctly predicted matches
    fp = 0  # Predicted match where none should exist
    fn = 0  # Missed match
    tn = 0  # Correctly predicted non-match

    # Evaluate ALL N1 × N2 node pairs
    for i in range(N1):
        for j in range(N2):
            pred_is_match = (hard_assign[i, j] == 1)
            gt_is_match = (P_gt[i, j] == 1)

            if gt_is_match and pred_is_match:
                tp += 1
            elif gt_is_match and not pred_is_match:
                fn += 1
            elif not gt_is_match and pred_is_match:
                fp += 1
            else:
                tn += 1

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }

def permutation_loss(S_pred, P_gt):
    """
    Permutation Loss - Binary Cross-Entropy applied element-wise.
    """
    N1, N2 = P_gt.shape

    # Take only real columns (first N2)
    S_real = S_pred[:, :N2]

    # BCE loss
    loss = -(P_gt * torch.log(S_real + 1e-8) + (1 - P_gt) * torch.log(1 - S_real + 1e-8)).mean()

    return loss

def evaluate(model, loader, device, verbose=True):
    """
    Evaluate model and return metrics.
    """
    model.eval()
    total_loss = 0
    all_metrics = []

    iterator = tqdm(loader, desc="Evaluating") if verbose else loader

    with torch.no_grad():
        for batch1, batch2, perm_list in iterator:
            # Move entire batch to device (not individual graphs)
            batch1 = batch1.to(device)
            batch2 = batch2.to(device)
            perm_list = [p.to(device) for p in perm_list]

            # Pass entire batches to model
            S_pred_list, _ = model(batch1, batch2)

            # Compute metrics for each graph pair in the batch
            for i, S_pred in enumerate(S_pred_list):
                P_gt = perm_list[i]
                loss = permutation_loss(S_pred, P_gt)
                metrics = compute_metrics(S_pred, P_gt)
                metrics['loss'] = loss.item()

                all_metrics.append(metrics)
                total_loss += loss.item()

    avg_metrics = {
        'loss': total_loss / len(all_metrics),
        'precision': np.mean([m['precision'] for m in all_metrics]),
        'recall': np.mean([m['recall'] for m in all_metrics]),
        'f1': np.mean([m['f1'] for m in all_metrics]),
        'accuracy': np.mean([m['accuracy'] for m in all_metrics])
    }

    return avg_metrics, all_metrics

## Load Preprocessed Dataset

In [ ]:
DATA_PATH = "/content/msd_data"

train_pairs = deserialize_graph_matching_dataset(DATA_PATH, "train_dataset.pkl")
val_pairs = deserialize_graph_matching_dataset(DATA_PATH, "valid_dataset.pkl")
test_pairs = deserialize_graph_matching_dataset(DATA_PATH, "test_dataset.pkl")

# Load original and noise graphs for visualization
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

with open(ORIGINAL_PATH, 'rb') as f:
    original_graphs_nx = pickle.load(f)
print(f"Loaded {len(original_graphs_nx)} original A-graphs")

with open(NOISE_PATH, 'rb') as f:
    noise_graphs_nx = pickle.load(f)
print(f"Loaded {len(noise_graphs_nx)} noise S-graphs")

print(f"\nDataset statistics:")
print(f"  Train: {len(train_pairs)} pairs")
print(f"  Validation: {len(val_pairs)} pairs")
print(f"  Test: {len(test_pairs)} pairs")

# Verify partial matching
sample_g1, sample_g2, sample_P = train_pairs[0]
print(f"\nPartial Matching Verification:")
print(f"  A-graph nodes: {sample_g1.x.shape[0]}")
print(f"  S-graph nodes: {sample_g2.x.shape[0]}")
print(f"  Matches: {sample_P.sum().item()} (should equal S-graph nodes)")

# Compute normalization statistics (mean, std) from training set only
print("\nComputing normalization statistics...")
mean, std = compute_mean_std(train_pairs)
print(f"Mean: {mean.tolist()}")
print(f"Std: {std.tolist()}")

# Normalize datasets
train_pairs_norm = normalize_data_pairs(train_pairs, mean, std)
val_pairs_norm = normalize_data_pairs(val_pairs, mean, std)
test_pairs_norm = normalize_data_pairs(test_pairs, mean, std)

# Create datasets and dataloaders
train_dataset = GraphMatchingDataset(train_pairs_norm)
val_dataset = GraphMatchingDataset(val_pairs_norm)
test_dataset = GraphMatchingDataset(test_pairs_norm)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pyg_matching)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pyg_matching)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pyg_matching)

print(f"\nDataLoaders created with batch size {BATCH_SIZE}")

# Load the first sample from raw and normalized data
sample_raw_g1, sample_raw_g2, sample_raw_P = train_pairs[0]
sample_norm_g1, sample_norm_g2, sample_norm_P = train_pairs_norm[0]

print(f"\nRaw Data - A-graph (g1):")
print(f"  Type: {type(sample_raw_g1)}")
print(f"  Attributes: {dir(sample_raw_g1)}")
print(f"  x shape: {sample_raw_g1.x.shape} (nodes × features)")
print(f"  edge_index shape: {sample_raw_g1.edge_index.shape} (2 × edges)")
print(f"  Has node_names: {hasattr(sample_raw_g1, 'node_names')}")
if hasattr(sample_raw_g1, 'node_names'):
    print(f"  node_names (first 5): {sample_raw_g1.node_names[:5]}")
print(f"  Has name: {hasattr(sample_raw_g1, 'name')}")
if hasattr(sample_raw_g1, 'name'):
    print(f"  name: {sample_raw_g1.name}")

print(f"\nRaw Data - S-graph (g2):")
print(f"  Type: {type(sample_raw_g2)}")
print(f"  x shape: {sample_raw_g2.x.shape}")
print(f"  edge_index shape: {sample_raw_g2.edge_index.shape}")
if hasattr(sample_raw_g2, 'node_names'):
    print(f"  node_names (first 5): {sample_raw_g2.node_names[:5]}")
if hasattr(sample_raw_g2, 'name'):
    print(f"  name: {sample_raw_g2.name}")

print(f"\nGround Truth Permutation Matrix (P):")
print(f"  Shape: {sample_raw_P.shape}")
print(f"  Type: {sample_raw_P.dtype}")
print(f"  Total matches: {sample_raw_P.sum().item()}")
print(f"  Row sums (first 10): {sample_raw_P.sum(dim=1)[:10].tolist()}")
print(f"  Column sums (first 10): {sample_raw_P.sum(dim=0)[:10].tolist()}")

# Feature names based on paper Table I
feature_names = [
    'Type_Room',      # [1,0] for room
    'Type_WS',        # [0,1] for wall segment
    'Centroid_X',     # x coordinate
    'Centroid_Y',     # y coordinate
    'Normal_X',       # outward-facing normal x
    'Normal_Y',       # outward-facing normal y
    'Segment_Length'  # length (-1 for rooms)
]

print("\nA-GRAPH (g1) Node Features - First 5 nodes:")
print(f"{'Node':<6} {'Feature':<15} {'Raw Value':<15} {'Normalized':<15} {'Diff':<15}")

for node_idx in range(min(5, sample_raw_g1.x.shape[0])):
    for feat_idx, feat_name in enumerate(feature_names):
        raw_val = sample_raw_g1.x[node_idx, feat_idx].item()
        norm_val = sample_norm_g1.x[node_idx, feat_idx].item()
        diff = norm_val - raw_val
        print(f"{node_idx:<6} {feat_name:<15} {raw_val:<15.6f} {norm_val:<15.6f} {diff:<15.6f}")
    print("-"*75)

print("\nS-GRAPH (g2) Node Features - First 5 nodes:")
print(f"{'Node':<6} {'Feature':<15} {'Raw Value':<15} {'Normalized':<15} {'Diff':<15}")

for node_idx in range(min(5, sample_raw_g2.x.shape[0])):
    for feat_idx, feat_name in enumerate(feature_names):
        raw_val = sample_raw_g2.x[node_idx, feat_idx].item()
        norm_val = sample_norm_g2.x[node_idx, feat_idx].item()
        diff = norm_val - raw_val
        print(f"{node_idx:<6} {feat_name:<15} {raw_val:<15.6f} {norm_val:<15.6f} {diff:<15.6f}")
    print("-"*75)

# Compute statistics for raw data
all_raw_features = []
for g1, g2, _ in train_pairs[:100]:  # First 100 samples
    all_raw_features.append(g1.x)
    all_raw_features.append(g2.x)
all_raw_features = torch.cat(all_raw_features, dim=0)

# Compute statistics for normalized data
all_norm_features = []
for g1, g2, _ in train_pairs_norm[:100]:
    all_norm_features.append(g1.x)
    all_norm_features.append(g2.x)
all_norm_features = torch.cat(all_norm_features, dim=0)

print(f"\n{'Feature':<15} {'Raw Mean':<12} {'Raw Std':<12} {'Norm Mean':<12} {'Norm Std':<12}")

for feat_idx, feat_name in enumerate(feature_names):
    raw_mean = all_raw_features[:, feat_idx].mean().item()
    raw_std = all_raw_features[:, feat_idx].std().item()
    norm_mean = all_norm_features[:, feat_idx].mean().item()
    norm_std = all_norm_features[:, feat_idx].std().item()
    print(f"{feat_name:<15} {raw_mean:<12.4f} {raw_std:<12.4f} {norm_mean:<12.4f} {norm_std:<12.4f}")

print(f"\nA-GRAPH Edges:")
print(f"  Raw edge_index shape: {sample_raw_g1.edge_index.shape}")
print(f"  Normalized edge_index shape: {sample_norm_g1.edge_index.shape}")
print(f"  Are edges identical? {torch.all(sample_raw_g1.edge_index == sample_norm_g1.edge_index).item()}")

# Show first 10 edges
print(f"\n  First 10 edges (raw):")
for i in range(min(10, sample_raw_g1.edge_index.shape[1])):
    u, v = sample_raw_g1.edge_index[0, i].item(), sample_raw_g1.edge_index[1, i].item()
    print(f"    Edge {i}: {u} → {v}")

print(f"\n  First 10 edges (normalized - should be identical):")
for i in range(min(10, sample_norm_g1.edge_index.shape[1])):
    u, v = sample_norm_g1.edge_index[0, i].item(), sample_norm_g1.edge_index[1, i].item()
    print(f"    Edge {i}: {u} → {v}")

# Check edge types if available
print(f"\n  Edge types: Normalization does NOT change edge indices - they remain identical")

print(f"\nPermutation Matrix P:")
print(f"  Raw P shape: {sample_raw_P.shape}")
print(f"  Normalized P shape: {sample_norm_P.shape}")
print(f"  Are P matrices identical? {torch.all(sample_raw_P == sample_norm_P).item()}")

# Show non-zero entries
nonzero_indices = torch.where(sample_raw_P > 0.5)
print(f"\n  Non-zero entries (first 10 matches):")
for i in range(min(10, len(nonzero_indices[0]))):
    row = nonzero_indices[0][i].item()
    col = nonzero_indices[1][i].item()
    print(f"    Match {i}: A-graph node {row} ↔ S-graph node {col}")

print(f"\n  P matrix sparsity: {sample_raw_P.sum().item() / (sample_raw_P.shape[0] * sample_raw_P.shape[1]) * 100:.2f}%")

print(f"\nNormalization formula: x_norm = (x - mean) / std")
print(f"\nMean (computed from training set): {mean.tolist()}")
print(f"Std (computed from training set): {std.tolist()}")

# Demonstrate transformation on a sample node
sample_node_idx = 0
print(f"\nExample: Node {sample_node_idx} from A-graph")
for feat_idx, feat_name in enumerate(feature_names):
    raw_val = sample_raw_g1.x[sample_node_idx, feat_idx].item()
    norm_val = sample_norm_g1.x[sample_node_idx, feat_idx].item()
    computed_norm = (raw_val - mean[feat_idx].item()) / (std[feat_idx].item() + 1e-8)
    print(f"  {feat_name}: raw={raw_val:.4f} → norm={norm_val:.4f} (computed={computed_norm:.4f})")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
feature_idx_to_plot = [0, 2, 4, 6]  # Type, Centroid X, Normal X, Length

for idx, feat_idx in enumerate(feature_idx_to_plot):
    # Raw distribution
    ax = axes[0, idx]
    raw_values = all_raw_features[:, feat_idx].numpy()
    ax.hist(raw_values, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(f'Raw: {feature_names[feat_idx]}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.axvline(raw_values.mean(), color='red', linestyle='--', label=f'Mean={raw_values.mean():.2f}')
    ax.legend()

    # Normalized distribution
    ax = axes[1, idx]
    norm_values = all_norm_features[:, feat_idx].numpy()
    ax.hist(norm_values, bins=50, edgecolor='black', alpha=0.7, color='coral')
    ax.set_title(f'Normalized: {feature_names[feat_idx]}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.axvline(norm_values.mean(), color='red', linestyle='--', label=f'Mean={norm_values.mean():.2f}')
    ax.legend()

plt.suptitle('Feature Distributions: Raw vs Normalized', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Check for NaN or Inf
print("\nChecking for NaN/Inf values:")
print(f"  Raw A-graph has NaN: {torch.isnan(sample_raw_g1.x).any().item()}")
print(f"  Raw A-graph has Inf: {torch.isinf(sample_raw_g1.x).any().item()}")
print(f"  Normalized A-graph has NaN: {torch.isnan(sample_norm_g1.x).any().item()}")
print(f"  Normalized A-graph has Inf: {torch.isinf(sample_norm_g1.x).any().item()}")

# Check for constant features
print(f"\nChecking for constant features (std=0):")
for feat_idx, feat_name in enumerate(feature_names):
    raw_std = all_raw_features[:, feat_idx].std().item()
    norm_std = all_norm_features[:, feat_idx].std().item()
    if raw_std < 1e-6:
        print(f"  WARNING: {feat_name} has zero variance in raw data!")
    if norm_std < 1e-6:
        print(f"  WARNING: {feat_name} has zero variance after normalization!")

# Check if normalization is appropriate
print(f"\nNormalization assessment:")
print(f"  Mean should be ~0: actual mean = {all_norm_features.mean().item():.6f}")
print(f"  Std should be ~1: actual std = {all_norm_features.std().item():.6f}")

if abs(all_norm_features.mean().item()) > 0.1:
    print(f"  WARNING: Mean is far from 0! Normalization may be incorrect.")
if abs(all_norm_features.std().item() - 1) > 0.1:
    print(f"  WARNING: Std is far from 1! Normalization may be incorrect.")


## Dataset Assessment

### Check Data Format

In [ ]:
# Examine a single training sample
sample_g1, sample_g2, sample_P = train_pairs_norm[0]

print("\n1. Sample Data Shapes:")
print(f"   A-graph (reference): {sample_g1.x.shape}")
print(f"   S-graph (query): {sample_g2.x.shape}")
print(f"   Ground truth P matrix: {sample_P.shape}")

print("\n2. Ground Truth Statistics:")
print(f"   Total matches (sum of P): {sample_P.sum().item()}")
print(f"   Number of rows (A-graph nodes): {sample_P.shape[0]}")
print(f"   Number of cols (S-graph nodes): {sample_P.shape[1]}")
print(f"   Matches per A-graph node (row sums): {sample_P.sum(dim=1).tolist()[:10]}")
print(f"   Matches per S-graph node (col sums): {sample_P.sum(dim=0).tolist()[:10]}")

# Check if P is a proper permutation matrix for partial matching
row_sums = sample_P.sum(dim=1)
col_sums = sample_P.sum(dim=0)

print("\n3. Permutation Matrix Properties:")
print(f"   Rows with sum=1 (should match): {(row_sums == 1).sum().item()}/{sample_P.shape[0]}")
print(f"   Rows with sum=0 (no match): {(row_sums == 0).sum().item()}/{sample_P.shape[0]}")
print(f"   Columns with sum=1: {(col_sums == 1).sum().item()}/{sample_P.shape[1]}")
print(f"   Columns with sum=0: {(col_sums == 0).sum().item()}/{sample_P.shape[1]}")

print("\n4. Node Features Check:")
print(f"   A-graph features (first 3 nodes):")
for i in range(min(3, sample_g1.x.shape[0])):
    feat = sample_g1.x[i].tolist()
    node_type = "ROOM" if feat[0] > 0.5 else "WALL"
    print(f"     Node {i} ({node_type}): center=({feat[2]:.2f}, {feat[3]:.2f}), normal=({feat[4]:.2f}, {feat[5]:.2f}), len={feat[6]:.2f}")

print(f"\n   S-graph features (first 3 nodes):")
for i in range(min(3, sample_g2.x.shape[0])):
    feat = sample_g2.x[i].tolist()
    node_type = "ROOM" if feat[0] > 0.5 else "WALL"
    print(f"     Node {i} ({node_type}): center=({feat[2]:.2f}, {feat[3]:.2f}), normal=({feat[4]:.2f}, {feat[5]:.2f}), len={feat[6]:.2f}")

# Check if features are properly normalized
print("\n5. Feature Normalization Check:")
print(f"   A-graph mean: {sample_g1.x.mean(dim=0).tolist()}")
print(f"   A-graph std: {sample_g1.x.std(dim=0).tolist()}")
print(f"   Should be close to 0 mean and 1 std after normalization")

### Visualize Training Samples

In [ ]:
# Load original and noise graphs
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

with open(ORIGINAL_PATH, 'rb') as f:
    original_graphs_nx = pickle.load(f)
print(f"Loaded {len(original_graphs_nx)} original A-graphs")

with open(NOISE_PATH, 'rb') as f:
    noise_graphs_nx = pickle.load(f)
print(f"Loaded {len(noise_graphs_nx)} noise S-graphs")

# Get 5 training samples
num_samples = min(5, len(train_pairs))
print(f"\nVisualizing {num_samples} training samples...")

for idx in range(num_samples):
    g1_pyg, g2_pyg, P_gt = train_pairs[idx]

    # Ensure the PyG tensors have the required attributes
    print(f"SAMPLE {idx+1}")
    print(f"  A-graph nodes: {g1_pyg.x.shape[0]}")
    print(f"  S-graph nodes: {g2_pyg.x.shape[0]}")
    print(f"  Ground truth matches: {P_gt.sum().item()}")

    # Create visualization
    fig, ax = plot_two_graphs_with_matching(
        graphs_list=[g1_pyg, g2_pyg],
        gt_perm=P_gt,
        original_graphs=original_graphs_nx,
        pred_perm=P_gt,  # Use ground truth as prediction (all correct matches)
        noise_graphs=noise_graphs_nx,
        viz_rooms=True,
        viz_ws=True,
        viz_room_connection=True,
        viz_normals=False,
        viz_room_normals=False,
        match_display="all",
        title=f"Training Sample {idx+1}: Ground Truth Matching (A:{g1_pyg.x.shape[0]} → S:{g2_pyg.x.shape[0]})",
        save_path=None
    )
    plt.show()
    plt.close(fig)

## GNN Model Architecture Definitions

In [ ]:
@dataclass
class ModelParams:
    """Unified parameters for all GNN models."""

    # Data Parameters
    input_dim: int = 7
    batch_size: int = 8

    # GNN Architecture Parameters
    hidden_dim: int = 128
    output_dim: int = 16
    num_layers: int = 2
    num_heads: int = 4

    # Regularization Parameters
    dropout: float = 0.0001792177005695561
    attn_dropout: float = 0.00609918816039232

    # Sinkhorn Parameters
    sinkhorn_iterations: int = 73
    sinkhorn_tau: float = 0.999101821286028

    # Optimization Parameters
    learning_rate: float = 0.0023737917298792635
    weight_decay: float = 3.272922404797929e-05

    # Training Parameters
    num_epochs: int = 100
    patience: int = 10

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary."""
        return asdict(self)

    def to_json(self, filepath: str):
        """Save parameters to JSON file."""
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def from_json(cls, filepath: str) -> 'ModelParams':
        """Load parameters from JSON file."""
        with open(filepath, 'r') as f:
            data = json.load(f)
        return cls(**data)

    @classmethod
    def get_default(cls) -> 'ModelParams':
        """Get default parameters (optimal from paper author [Matteo Giorgi])."""
        return cls()

    @classmethod
    def get_paper_params(cls) -> 'ModelParams':
        """Get published paper parameters (non-optimal)."""
        return cls(
            hidden_dim=64,
            output_dim=32,
            dropout=0.15,
            attn_dropout=0.12,
            sinkhorn_iterations=20,
            sinkhorn_tau=1.0,
            learning_rate=0.001,
            weight_decay=5e-5,
            batch_size=16
        )

DEFAULT_PARAMS = ModelParams.get_default()

class FeatureHomogenizer(nn.Module):
    """
    MLP to homogenize heterogeneous node features.
    As described in paper Section III-A: two-layer MLP with ReLU and dropout.
    """
    def __init__(self, params: ModelParams = None):
        super().__init__()
        if params is None:
            params = DEFAULT_PARAMS

        self.mlp = nn.Sequential(
            nn.Linear(params.input_dim, params.hidden_dim),
            nn.ReLU(),
            nn.Dropout(params.dropout),
            nn.Linear(params.hidden_dim, params.hidden_dim),
            nn.ReLU(),
            nn.Dropout(params.dropout)
        )

    def forward(self, x):
        return self.mlp(x)

class BaseGNNEncoder(nn.Module, ABC):
    """Base class for all GNN encoders."""
    def __init__(self, params: ModelParams = None):
        super().__init__()
        if params is None:
            params = DEFAULT_PARAMS

        self.params = params
        self.homogenizer = FeatureHomogenizer(params)
        self.output_dim = params.output_dim
        self.dropout = params.dropout

    @abstractmethod
    def forward(self, data):
        pass

class GCNEncoder(BaseGNNEncoder):
    """GCN encoder matching GATv2 baseline flow."""
    def __init__(self, params: ModelParams = None):
        super().__init__(params)
        if params is None:
            params = DEFAULT_PARAMS

        self.conv1 = GCNConv(params.hidden_dim, params.hidden_dim)
        self.conv2 = GCNConv(params.hidden_dim, params.output_dim)
        self.dropout = nn.Dropout(p=params.dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.homogenizer(x)

        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)

        return x

class GraphSAGEEncoder(BaseGNNEncoder):
    """GraphSAGE encoder matching GATv2 baseline flow."""
    def __init__(self, params: ModelParams = None):
        super().__init__(params)
        if params is None:
            params = DEFAULT_PARAMS

        self.conv1 = SAGEConv(params.hidden_dim, params.hidden_dim)
        self.conv2 = SAGEConv(params.hidden_dim, params.output_dim)
        self.dropout = nn.Dropout(p=params.dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.homogenizer(x)

        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)

        return x

class GINEncoder(BaseGNNEncoder):
    """GIN encoder matching GATv2 baseline flow."""
    def __init__(self, params: ModelParams = None):
        super().__init__(params)
        if params is None:
            params = DEFAULT_PARAMS

        mlp1 = nn.Sequential(
            nn.Linear(params.hidden_dim, params.hidden_dim),
            nn.ReLU(),
            nn.Linear(params.hidden_dim, params.hidden_dim)
        )
        mlp2 = nn.Sequential(
            nn.Linear(params.hidden_dim, params.hidden_dim),
            nn.ReLU(),
            nn.Linear(params.hidden_dim, params.output_dim)
        )

        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)
        self.dropout = nn.Dropout(p=params.dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.homogenizer(x)

        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)

        return x

class GraphTransformerEncoder(BaseGNNEncoder):
    """
    Graph Transformer encoder matching GATv2 baseline flow.
    """
    def __init__(self, params: ModelParams = None):
        super().__init__(params)
        if params is None:
            params = DEFAULT_PARAMS

        # First layer: hidden_dim -> hidden_dim
        self.conv1 = TransformerConv(
            in_channels=params.hidden_dim,
            out_channels=params.hidden_dim,
            heads=params.num_heads,
            dropout=params.attn_dropout,
            concat=False  # Average heads instead of concatenating
        )

        # Second layer: hidden_dim -> output_dim
        self.conv2 = TransformerConv(
            in_channels=params.hidden_dim,
            out_channels=params.output_dim,
            heads=params.num_heads,
            dropout=params.attn_dropout,
            concat=False
        )

        self.dropout = nn.Dropout(p=params.dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.homogenizer(x)

        x = self.dropout(x)

        # First Graph Transformer layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        # Second Graph Transformer layer
        x = self.conv2(x, edge_index)

        return x

class GATv2Encoder(BaseGNNEncoder):
    """
    GATv2 encoder matching PAPER specifications (Section III-B) with OPTIMAL hyperparameters.
    - Both layers: 4 heads, concat=False
    - hidden_dim = 128, output_dim = 16
    - dropout = 0.00018, attn_dropout = 0.0061
    GATv2 encoder matching MLPGATv2Sinkhorn architecture
    """
    def __init__(self, params: ModelParams = None):
        super().__init__(params)
        if params is None:
            params = DEFAULT_PARAMS

        self.conv1 = GATv2Conv(
            params.hidden_dim, params.hidden_dim,
            heads=4,
            dropout=params.attn_dropout,
            concat=False
        )

        self.conv2 = GATv2Conv(
            params.hidden_dim, params.output_dim,
            heads=4,
            dropout=params.attn_dropout,
            concat=False
        )

        self.dropout = nn.Dropout(p=params.dropout)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.homogenizer(x)

        x = self.dropout(x)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)

        return x

class GraphMatcher(nn.Module):
    """
    Graph matching model with Sinkhorn.
    Matches MatchingModel_MLPGATv2Sinkhorn architecture
    """
    def __init__(self, encoder, params: ModelParams = None):
        super().__init__()
        self.encoder = encoder

        if params is None:
            params = DEFAULT_PARAMS

        self.params = params
        self.sinkhorn_max_iter = params.sinkhorn_iterations
        self.sinkhorn_tau = params.sinkhorn_tau
        self.inst_norm = nn.InstanceNorm2d(1, affine=True)

    def forward(self, batch1, batch2, perm_list=None, batch_idx1=None, batch_idx2=None, inference=False):
        """
        Forward pass for batched graph matching.
        """
        device = next(self.parameters()).device

        # Move data to device
        x1, edge1 = batch1.x.to(device), batch1.edge_index.to(device)
        x2, edge2 = batch2.x.to(device), batch2.edge_index.to(device)

        # Get batch indices
        batch_idx1 = batch1.batch.to(device) if batch_idx1 is None else batch_idx1.to(device)
        batch_idx2 = batch2.batch.to(device) if batch_idx2 is None else batch_idx2.to(device)

        # Encode both graphs
        h1 = self.encoder(batch1)
        h2 = self.encoder(batch2)

        B = batch_idx1.max().item() + 1
        perm_pred_list = []
        all_embeddings = []

        for b in range(B):
            h1_b = h1[batch_idx1 == b]   # [n1, d]
            h2_b = h2[batch_idx2 == b]   # [n2, d]
            N1, N2 = h1_b.size(0), h2_b.size(0)

            # Affinity matrix
            sim = torch.matmul(h1_b, h2_b.T)  # [n1, n2]
            sim_batched = sim.unsqueeze(0).unsqueeze(1)  # [1,1,n1,n2]
            sim_normed = self.inst_norm(sim_batched).squeeze(1)  # [1,n1,n2]

            # Handle partial matching (N1 != N2)
            transposed = N1 > N2

            if transposed:
                sim_input = sim_normed.transpose(-2, -1)  # [1, n2, n1]
                nr = torch.tensor([N2], dtype=torch.long, device=device)
                nc = torch.tensor([N1], dtype=torch.long, device=device)
            else:
                sim_input = sim_normed  # [1, n1, n2]
                nr = torch.tensor([N1], dtype=torch.long, device=device)
                nc = torch.tensor([N2], dtype=torch.long, device=device)

            S = pygmtools.sinkhorn(
                sim_input,
                n1=nr, n2=nc,
                dummy_row=(N1 != N2),
                max_iter=self.sinkhorn_max_iter,
                tau=self.sinkhorn_tau
            )

            if transposed:
                S = S.transpose(-2, -1)  # rollback to [1, n1, n2]

            perm_pred_list.append(S.squeeze(0))  # [n1, n2]
            all_embeddings.append((h1_b, h2_b))

        return perm_pred_list, all_embeddings

    def get_hard_assignment(self, S, N2):
        """Extract hard assignment using Hungarian algorithm."""
        # S is [n1, n2]
        return pygmtools.hungarian(S)

class MatchingModel_MLPGATv2Sinkhorn(nn.Module):
    """
    MLP + GATv2 + Sinkhorn model for graph matching.
    Modified to use params object like our other models.
    """
    def __init__(self, params: ModelParams = None):
        super().__init__()

        if params is None:
            params = DEFAULT_PARAMS

        self.params = params

        # Extract parameters
        in_dim = params.input_dim
        hidden_dim = params.hidden_dim
        out_dim = params.output_dim
        sinkhorn_max_iter = params.sinkhorn_iterations
        sinkhorn_tau = params.sinkhorn_tau
        attention_dropout = params.attn_dropout
        dropout_emb = params.dropout
        num_layers = params.num_layers
        heads = params.num_heads

        # MLP for initial node feature transformation
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_emb),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_emb)
        )

        self.gnn = nn.ModuleList()
        dims = [hidden_dim] * num_layers + [out_dim]
        for i in range(num_layers):
            # Always average the heads so the feature-dim stays dims[i+1]
            self.gnn.append(
                GATv2Conv(dims[i], dims[i+1],
                            heads=heads, concat=False,
                            dropout=attention_dropout)
            )

        self.dropout = nn.Dropout(p=dropout_emb)
        self.inst_norm = nn.InstanceNorm2d(1, affine=True)
        self.sinkhorn_max_iter = sinkhorn_max_iter
        self.sinkhorn_tau = sinkhorn_tau

    def encode(self, x, edge_index):
        for i, conv in enumerate(self.gnn):
            x = conv(x, edge_index)
            if i < len(self.gnn) - 1:
                x = F.relu(x)
                x = self.dropout(x)
        return x

    def forward(self, batch1, batch2, perm_list=None, batch_idx1=None, batch_idx2=None, inference=False):
        device = next(self.parameters()).device
        x1, edge1 = batch1.x.to(device), batch1.edge_index.to(device)
        x2, edge2 = batch2.x.to(device), batch2.edge_index.to(device)
        perm_list = [p.to(device) for p in perm_list] if perm_list is not None else None

        batch_idx1 = batch1.batch.to(device) if batch_idx1 is None else batch_idx1.to(device)
        batch_idx2 = batch2.batch.to(device) if batch_idx2 is None else batch_idx2.to(device)

        # Apply MLP before GNN
        h1 = self.mlp(x1)
        h2 = self.mlp(x2)
        h1 = self.encode(h1, edge1)
        h2 = self.encode(h2, edge2)

        B = batch_idx1.max().item() + 1
        perm_pred_list = []
        all_embeddings = []

        for b in range(B):
            h1_b = h1[batch_idx1 == b]   # [n1, d]
            h2_b = h2[batch_idx2 == b]   # [n2, d]
            N1, N2 = h1_b.size(0), h2_b.size(0)

            # affinity matrix + normalization + sinkhorn
            sim = torch.matmul(h1_b, h2_b.T) # [n1, n2]
            sim_batched = sim.unsqueeze(0).unsqueeze(1) # [1,1,n1,n2]
            sim_normed = self.inst_norm(sim_batched).squeeze(1) # [1,n1,n2]

            # g1 -> A-graph
            # g2 -> S-graph (partial)
            transposed = N1 > N2

            if transposed:
                # traspose to use dummy_row
                sim_input = sim_normed.transpose(-2, -1)   # [1, n2, n1]
                nr = torch.tensor([N2], dtype=torch.long, device=device)
                nc = torch.tensor([N1], dtype=torch.long, device=device)
            else:
                sim_input = sim_normed                     # [1, n1, n2]
                nr = torch.tensor([N1], dtype=torch.long, device=device)
                nc = torch.tensor([N2], dtype=torch.long, device=device)

            S = pygmtools.sinkhorn(
                sim_input,
                n1=nr, n2=nc,
                dummy_row=(N1 != N2),
                max_iter=self.sinkhorn_max_iter,
                tau=self.sinkhorn_tau
            )

            if transposed:
                S = S.transpose(-2, -1)   # rollback to [1, n1, n2]

            perm_pred_list.append(S.squeeze(0))  # [n1, n2]
            all_embeddings.append((h1_b, h2_b))

        return perm_pred_list, all_embeddings

## Baseline Model (GATv2 Encoder) ([Brody et al., 2021](https://arxiv.org/abs/2105.14491))

### Train the Baseline Model (GATv2)

#### Model Inspection

In [ ]:
# Load optimal hyperparameters
optimal_params = ModelParams.get_default()

print("\nOptimal Hyperparameters:")
for key, value in asdict(optimal_params).items():
    print(f"  {key}: {value}")

# Initialize model with optimal parameters
DEFAULT_PARAMS = ModelParams.get_default()

encoder = GATv2Encoder(optimal_params)
GATv2_model = GraphMatcher(encoder, optimal_params).to(device)

# Break down parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
homogenizer_params = sum(p.numel() for p in encoder.homogenizer.parameters())
gat_params = sum(p.numel() for p in encoder.conv1.parameters()) + sum(p.numel() for p in encoder.conv2.parameters())

print(f"\nParameter Breakdown:")
print(f"  Encoder total: {encoder_params:,}")
print(f"    - MLP Homogenizer: {homogenizer_params:,}")
print(f"    - GATv2 layers: {gat_params:,}")
print(f"\nGATv2 Model Architecture:")
print(GATv2_model)

In [ ]:
shared_model = MatchingModel_MLPGATv2Sinkhorn(params=optimal_params)
print("\nShared Model Architecture:")
print(shared_model)

#### Model Training

In [ ]:
# Optimizer with optimal learning rate
optimizer = torch.optim.AdamW(
    GATv2_model.parameters(),
    lr=optimal_params.learning_rate,
    weight_decay=optimal_params.weight_decay
)

# Update dataloader with optimal batch size
BATCH_SIZE_OPTIMAL = optimal_params.batch_size

# Recreate dataloaders with optimal batch size
train_loader_optimal = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=True,
    collate_fn=collate_pyg_matching
)
val_loader_optimal = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)
test_loader_optimal = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)

print(f"\nDataLoaders recreated with batch_size={BATCH_SIZE_OPTIMAL}")

# Check for existing checkpoint to continue training
checkpoint_path = '/content/pretrained_models/best_GATv2_model.pt'
start_epoch = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
val_f1_scores = []
val_precision = []
val_recall = []

if os.path.exists(checkpoint_path):
    print(f"\nFound existing checkpoint at {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        GATv2_model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        train_losses = checkpoint.get('train_losses', [])
        val_losses = checkpoint.get('val_losses', [])
        val_f1_scores = checkpoint.get('val_f1_scores', [])

        print(f"  Resuming from epoch {start_epoch}")
        print(f"  Previous best loss: {best_val_loss:.4f}")
    except Exception as e:
        print(f"  Could not load checkpoint: {e}")
        print(f"  Starting fresh training")
        start_epoch = 0
else:
    print(f"\nNo xisting checkpoint found. Starting fresh training.")

NUM_EPOCHS = 46
PATIENCE = 5

patience_counter = 0

print(f"\nStarting training from epoch {start_epoch + 1}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    # Training Phase
    GATv2_model.train()
    epoch_train_loss = 0
    num_batches = 0

    for batch1, batch2, perm_list in tqdm(train_loader_optimal, desc=f"Epoch {epoch+1} Training"):
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)
        perm_list = [p.to(device) for p in perm_list]

        # Pass entire batches to model (returns list of S_pred for each graph in batch)
        S_pred_list, _ = GATv2_model(batch1, batch2)

        # Compute loss for each graph pair in the batch
        batch_loss = 0
        for i, S_pred in enumerate(S_pred_list):
            P_gt = perm_list[i]
            loss = permutation_loss(S_pred, P_gt)
            batch_loss += loss

        # Average loss over batch
        batch_loss = batch_loss / len(S_pred_list)

        # Backward pass
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        epoch_train_loss += batch_loss.item()
        num_batches += 1

    avg_train_loss = epoch_train_loss / num_batches if num_batches > 0 else 0
    train_losses.append(avg_train_loss)

    # Validation Phase
    val_metrics, val_detailed = evaluate(GATv2_model, val_loader_optimal, device)
    val_loss = val_metrics.get('loss', 0)
    val_losses.append(val_loss)
    val_f1_scores.append(val_metrics['f1'])
    val_precision.append(val_metrics['precision'])
    val_recall.append(val_metrics['recall'])

    current_lr = optimizer.param_groups[0]['lr']

    # Print progress every epoch
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val F1: {val_metrics['f1']:.4f} | "
          f"LR: {current_lr:.2e}")

    # Early Stopping (based on validation loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': GATv2_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_f1_scores': val_f1_scores,
            'hyperparams': optimal_params
        }, checkpoint_path)
        print(f"  → New best model! Val Loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best validation loss: {best_val_loss:.4f}")
            break

# Load best model
checkpoint = torch.load(checkpoint_path, weights_only=False)
GATv2_model.load_state_dict(checkpoint['model_state_dict'])

print("TRAINING COMPLETE")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(train_losses)}")

# Plot Training Curves
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot 1: Training and Validation Loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2, color='blue')
axes[0].plot(val_losses, label='Val Loss', linewidth=2, color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Validation F1 Score
axes[1].plot(val_f1_scores, label='Val F1 Score', linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle('Training Curves - GATv2 Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GATv2_baseline_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

### Test Set Evaluation

In [ ]:
# Load best model if not already loaded
if 'checkpoint' not in dir():
    checkpoint_path = '/content/pretrained_models/best_GATv2_model.pt'
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        GATv2_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from {checkpoint_path}")
    else:
        print(f"Warning: No checkpoint found at {checkpoint_path}")
        print("Using current model state.")

# Evaluate on test set
print("TEST SET EVALUATION")

# Data is already preprocessed and normalized.
test_metrics, test_detailed = evaluate(GATv2_model, test_loader, device)

print("TEST SET RESULTS")
print(f"{'Metric':<15} {'Our Result':<15} {'Paper Reported':<15}")
print(f"{'Precision':<15} {test_metrics['precision']:.4f}       {'0.85':<15}")
print(f"{'Recall':<15} {test_metrics['recall']:.4f}       {'0.85':<15}")
print(f"{'F1 Score':<15} {test_metrics['f1']:.4f}       {'0.85':<15} ← Primary")
print(f"{'Accuracy':<15} {test_metrics['accuracy']:.4f}       {'0.99':<15}")

# Measure inference time (as per research paper Section IV-A)
print("INFERENCE TIME MEASUREMENT")

GATv2_model.eval()
inference_times = []
WARMUP_SAMPLES = 20  # samples for warm-up
num_samples_to_measure = min(100, len(test_loader.dataset))

print(f"Warm-up samples: {WARMUP_SAMPLES}")
print(f"Measurement samples: {num_samples_to_measure}")

with torch.no_grad():
    samples_warmed = 0
    samples_measured = 0

    for batch1, batch2, perm_list in test_loader:
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)

        # Get number of graphs in this batch
        num_graphs = batch1.num_graphs if hasattr(batch1, 'num_graphs') else len(perm_list)

        for i in range(num_graphs):
            # Warm-up phase
            if samples_warmed < WARMUP_SAMPLES:
                S_pred_list, _ = GATv2_model(batch1, batch2)
                _ = S_pred_list[i]  # Just to ensure computation
                samples_warmed += 1
                continue

            # Measurement phase
            if samples_measured >= num_samples_to_measure:
                break

            # Synchronize for accurate timing
            if device.type == 'cuda':
                torch.cuda.synchronize()

            start_time = time.perf_counter()
            S_pred_list, _ = GATv2_model(batch1, batch2)
            _ = S_pred_list[i]  # Ensure computation is complete
            if device.type == 'cuda':
                torch.cuda.synchronize()

            inference_times.append(time.perf_counter() - start_time)
            samples_measured += 1

        if samples_measured >= num_samples_to_measure:
            break

avg_inference_time = np.mean(inference_times) * 1000  # ms
std_inference_time = np.std(inference_times) * 1000

print(f"\nAverage inference time: {avg_inference_time:.2f} ± {std_inference_time:.1f} ms")
print(f"Paper reported: 93 ms (0.093s)")
print(f"Speed relative to paper: {avg_inference_time/93:.2f}x")

# Distribution analysis
f1_scores = [m['f1'] for m in test_detailed]
print("\nF1 SCORE DISTRIBUTION ANALYSIS")
print(f"   Mean: {np.mean(f1_scores):.4f}")
print(f"   Median: {np.median(f1_scores):.4f}")
print(f"   Std: {np.std(f1_scores):.4f}")
print(f"   Min: {np.min(f1_scores):.4f}")
print(f"   Max: {np.max(f1_scores):.4f}")
print(f"   25th percentile: {np.percentile(f1_scores, 25):.4f}")
print(f"   75th percentile: {np.percentile(f1_scores, 75):.4f}")

# Plot test results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram of F1 scores
axes[0].hist(f1_scores, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(test_metrics['f1'], color='red', linestyle='--', linewidth=2,
                label=f'Mean: {test_metrics["f1"]:.3f}')
axes[0].axvline(np.median(f1_scores), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(f1_scores):.3f}')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of F1 Scores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(f1_scores, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_ylabel('F1 Score')
axes[1].set_title(f'F1 Score Distribution\nQ1: {np.percentile(f1_scores, 25):.3f}, '
                  f'Q2: {np.median(f1_scores):.3f}, Q3: {np.percentile(f1_scores, 75):.3f}')
axes[1].grid(True, alpha=0.3)

# Comparison bar chart
metrics_names = ['Precision', 'Recall', 'F1', 'Accuracy']
our_values = [test_metrics['precision'], test_metrics['recall'],
              test_metrics['f1'], test_metrics['accuracy']]
paper_values = [0.85, 0.85, 0.85, 0.99]

x = np.arange(len(metrics_names))
width = 0.35

axes[2].bar(x - width/2, our_values, width, label='Our Implementation', color='steelblue')
axes[2].bar(x + width/2, paper_values, width, label='Paper Reported', color='lightcoral')
axes[2].set_xlabel('Metric')
axes[2].set_ylabel('Score')
axes[2].set_title('Comparison: Our Results vs Research Paper')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].legend()
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GATv2_test_results.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("CLASSIFICATION REPORT")

# Collect all predictions and ground truth labels
all_pred_labels = []
all_true_labels = []

GATv2_model.eval()
with torch.no_grad():
    for batch1, batch2, perm_list in test_loader:
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)
        perm_list = [p.to(device) for p in perm_list]

        S_pred_list, _ = GATv2_model(batch1, batch2)

        for i, S_pred in enumerate(S_pred_list):
            P_gt = perm_list[i]
            N1, N2 = P_gt.shape

            S_real = S_pred[:, :N2]
            hard_assign = pygmtools.hungarian(S_real.unsqueeze(0)).squeeze(0)
            pred_labels = hard_assign.argmax(dim=1).cpu().numpy()
            true_labels = P_gt.argmax(dim=1).cpu().numpy()

            for j in range(N1):
                if P_gt[j].sum().item() > 0:
                    all_pred_labels.append(pred_labels[j])
                    all_true_labels.append(true_labels[j])

all_pred_labels = np.array(all_pred_labels)
all_true_labels = np.array(all_true_labels)

print(f"\nTotal matched predictions analyzed: {len(all_pred_labels)}")

# Get unique labels
unique_labels = sorted(np.unique(np.concatenate([all_true_labels, all_pred_labels])))
classes_to_show = unique_labels[:50]  # Show first 50 classes

# Generate classification report
report_dict = classification_report(all_true_labels, all_pred_labels, output_dict=True)

print(f"{'Class':<10} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 60)

for label in classes_to_show:
    d = report_dict[str(label)]
    print(f"S-{label:<7} {d['precision']:<12.4f} {d['recall']:<12.4f} {d['f1-score']:<12.4f} {d['support']:<10}")

if len(unique_labels) > 20:
    print(f"... and {len(unique_labels) - 20} more classes")

print(f"{'macro avg':<10} {report_dict['macro avg']['precision']:<12.4f} {report_dict['macro avg']['recall']:<12.4f} {report_dict['macro avg']['f1-score']:<12.4f} {report_dict['macro avg']['support']:<10.0f}")
print(f"{'weighted avg':<10} {report_dict['weighted avg']['precision']:<12.4f} {report_dict['weighted avg']['recall']:<12.4f} {report_dict['weighted avg']['f1-score']:<12.4f} {report_dict['weighted avg']['support']:<10.0f}")

### Visualize Model Predictions on Test Set

In [ ]:
# Load original and noise graphs
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

with open(ORIGINAL_PATH, 'rb') as f:
    original_graphs_nx = pickle.load(f)
print(f"Loaded {len(original_graphs_nx)} original A-graphs")

with open(NOISE_PATH, 'rb') as f:
    noise_graphs_nx = pickle.load(f)
print(f"Loaded {len(noise_graphs_nx)} noise S-graphs")

# Get mean and std for normalization
mean, std = compute_mean_std(train_pairs)
print(f"Using precomputed mean and std for normalization")

def normalize_on_fly(g):
    """Apply normalization while preserving attributes."""
    g_norm = Data(x=(g.x - mean) / (std + 1e-8), edge_index=g.edge_index)
    if hasattr(g, 'name'):
        g_norm.name = g.name
    if hasattr(g, 'node_names'):
        g_norm.node_names = g.node_names
    if hasattr(g, 'permutation'):
        g_norm.permutation = g.permutation
    return g_norm

# Get test samples (using original test_pairs, not normalized)
num_samples = min(20, len(test_pairs))
print(f"\nVisualizing {num_samples} test samples with predictions...")

with torch.no_grad():
    for idx in range(num_samples):
        g1_orig, g2_orig, P_gt = test_pairs[idx]

        # Normalize on the fly for model input
        g1_norm = normalize_on_fly(g1_orig)
        g2_norm = normalize_on_fly(g2_orig)

        # Wrap as batches
        batch1 = Batch.from_data_list([g1_norm]).to(device)
        batch2 = Batch.from_data_list([g2_norm]).to(device)
        P_gt = P_gt.to(device)

        # Get predictions
        S_pred_list, _ = GATv2_model(batch1, batch2)
        S_pred = S_pred_list[0]

        N1, N2 = P_gt.shape
        S_real = S_pred[:, :N2]

        # Hungarian assignment
        try:
            hard_assign = pygmtools.hungarian(
                S_real.unsqueeze(0),
                n1=torch.tensor([N1]),
                n2=torch.tensor([N2])
            ).squeeze(0)
        except:
            max_dim = max(N1, N2)
            S_padded = torch.zeros(max_dim, max_dim, device=S_real.device)
            S_padded[:N1, :N2] = S_real
            if N1 > N2:
                S_padded[N1:, :N2] = 1e-9
                S_padded[:N1, N2:] = 1e-9
            hard_assign_full = pygmtools.hungarian(S_padded.unsqueeze(0)).squeeze(0)
            hard_assign = hard_assign_full[:N1, :N2]

        # Evaluate all N1 × N2 pairs
        tp = fp = fn = tn = 0

        for i in range(N1):
            for j in range(N2):
                pred_match = (hard_assign[i, j] == 1)
                gt_match = (P_gt[i, j] == 1)

                if gt_match and pred_match:
                    tp += 1
                elif gt_match and not pred_match:
                    fn += 1
                elif not gt_match and pred_match:
                    fp += 1
                else:
                    tn += 1

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

        print(f"SAMPLE {idx+1}")
        print(f"  A-graph nodes: {g1_orig.x.shape[0]}")
        print(f"  S-graph nodes: {g2_orig.x.shape[0]}")
        print(f"  Ground truth matches: {P_gt.sum().item()}")
        print(f"  Total pairs evaluated: {N1 * N2}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

        # Print assignment details
        print(f"  Hungarian assignment matrix shape: {hard_assign.shape}")
        print(f"  Row sums (non-zero rows): {torch.where(hard_assign.sum(dim=1) > 0)[0].tolist()}")
        print(f"  Column sums: {hard_assign.sum(dim=0).tolist()}")

        # Check each ground truth match
        for i in range(N1):
            if (P_gt[i] > 0.5).any():
                gt_col = P_gt[i].argmax().item()
                pred_col = hard_assign[i].argmax().item() if hard_assign[i].sum() > 0 else -1
                print(f"    A-node {i}: GT→S-node {gt_col}, Pred→S-node {pred_col}, Correct: {pred_col == gt_col}")

        # Create visualization
        print(f"\n  Generating visualization...")
        fig, ax = plot_two_graphs_with_matching(
            graphs_list=[g1_orig, g2_orig],
            gt_perm=P_gt.cpu(),
            original_graphs=original_graphs_nx,
            pred_perm=hard_assign.cpu(),
            noise_graphs=noise_graphs_nx,
            viz_rooms=True,
            viz_ws=True,
            viz_room_connection=True,
            viz_normals=False,
            viz_room_normals=False,
            match_display="all",
            title=f"Test Sample {idx+1}: Predictions (F1={f1:.3f}) | ✓{tp} Correct, ✗{fp} Wrong",
            save_path=None
        )
        plt.show()
        plt.close(fig)

## GCN Encoder Model ([Kipf et al., 2017](https://arxiv.org/abs/1609.02907))

### Train GCN Model

In [ ]:
# Load optimal hyperparameters
optimal_params = ModelParams.get_default()

print("\nOptimal Hyperparameters:")
for key, value in asdict(optimal_params).items():
    print(f"  {key}: {value}")

# Initialize model with optimal parameters
DEFAULT_PARAMS = ModelParams.get_default()

encoder = GCNEncoder(optimal_params)
gcn_model = GraphMatcher(encoder, optimal_params).to(device)

# Break down parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
homogenizer_params = sum(p.numel() for p in encoder.homogenizer.parameters())
gat_params = sum(p.numel() for p in encoder.conv1.parameters()) + sum(p.numel() for p in encoder.conv2.parameters())

print(f"\nParameter Breakdown:")
print(f"  Encoder total: {encoder_params:,}")
print(f"    - MLP Homogenizer: {homogenizer_params:,}")
print(f"    - GCN layers: {gat_params:,}")
print(f"\nGCN Model Architecture:")
print(gcn_model)

# Optimizer with optimal learning rate
optimizer = torch.optim.AdamW(
    gcn_model.parameters(),
    lr=optimal_params.learning_rate,
    weight_decay=optimal_params.weight_decay
)

# Update dataloader with optimal batch size
BATCH_SIZE_OPTIMAL = optimal_params.batch_size

# Recreate dataloaders with optimal batch size
train_loader_optimal = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=True,
    collate_fn=collate_pyg_matching
)
val_loader_optimal = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)
test_loader_optimal = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)

print(f"\nDataLoaders recreated with batch_size={BATCH_SIZE_OPTIMAL}")

# Check for existing checkpoint to continue training
checkpoint_path = '/content/pretrained_models/best_GCN_model.pt'
start_epoch = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
val_f1_scores = []
val_precision = []
val_recall = []

if os.path.exists(checkpoint_path):
    print(f"\nFound existing checkpoint at {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        gcn_model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        train_losses = checkpoint.get('train_losses', [])
        val_losses = checkpoint.get('val_losses', [])
        val_f1_scores = checkpoint.get('val_f1_scores', [])

        print(f"  Resuming from epoch {start_epoch}")
        print(f"  Previous best loss: {best_val_loss:.4f}")
    except Exception as e:
        print(f"  Could not load checkpoint: {e}")
        print(f"  Starting fresh training")
        start_epoch = 0
else:
    print(f"\nNo existing checkpoint found. Starting fresh training.")

NUM_EPOCHS = 63
PATIENCE = 5

patience_counter = 0

print(f"\nStarting training from epoch {start_epoch + 1}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    # Training Phase
    gcn_model.train()
    epoch_train_loss = 0
    num_batches = 0

    for batch1, batch2, perm_list in tqdm(train_loader_optimal, desc=f"Epoch {epoch+1} Training"):
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)
        perm_list = [p.to(device) for p in perm_list]

        # Pass entire batches to model (returns list of S_pred for each graph in batch)
        S_pred_list, _ = gcn_model(batch1, batch2)

        # Compute loss for each graph pair in the batch
        batch_loss = 0
        for i, S_pred in enumerate(S_pred_list):
            P_gt = perm_list[i]
            loss = permutation_loss(S_pred, P_gt)
            batch_loss += loss

        # Average loss over batch
        batch_loss = batch_loss / len(S_pred_list)

        # Backward pass
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        epoch_train_loss += batch_loss.item()
        num_batches += 1

    avg_train_loss = epoch_train_loss / num_batches if num_batches > 0 else 0
    train_losses.append(avg_train_loss)

    # Validation Phase
    val_metrics, val_detailed = evaluate(gcn_model, val_loader_optimal, device)
    val_loss = val_metrics.get('loss', 0)
    val_losses.append(val_loss)
    val_f1_scores.append(val_metrics['f1'])
    val_precision.append(val_metrics['precision'])
    val_recall.append(val_metrics['recall'])

    current_lr = optimizer.param_groups[0]['lr']

    # Print progress every epoch
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val F1: {val_metrics['f1']:.4f} | "
          f"LR: {current_lr:.2e}")

    # Early Stopping (based on validation loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': gcn_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_f1_scores': val_f1_scores,
            'hyperparams': optimal_params
        }, checkpoint_path)
        print(f"  → New best model! Val Loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best validation loss: {best_val_loss:.4f}")
            break

# Load best model
checkpoint = torch.load(checkpoint_path, weights_only=False)
gcn_model.load_state_dict(checkpoint['model_state_dict'])

print("TRAINING COMPLETE")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(train_losses)}")

# Plot Training Curves
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot 1: Training and Validation Loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2, color='blue')
axes[0].plot(val_losses, label='Val Loss', linewidth=2, color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Validation F1 Score
axes[1].plot(val_f1_scores, label='Val F1 Score', linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle('Training Curves - GCN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GCN_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

### Test Set Evaluation

In [ ]:
# Load best model if not already loaded
if 'checkpoint' not in dir():
    checkpoint_path = '/content/pretrained_models/best_GCN_model.pt'
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        gcn_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from {checkpoint_path}")
    else:
        print(f"Warning: No checkpoint found at {checkpoint_path}")
        print("Using current model state.")

# Evaluate on test set
print("TEST SET EVALUATION")

# Data is already preprocessed and normalized.
test_metrics, test_detailed = evaluate(gcn_model, test_loader, device)

print("TEST SET RESULTS")
print(f"{'Metric':<15} {'Our Result':<15} {'Paper Reported':<15}")
print(f"{'Precision':<15} {test_metrics['precision']:.4f}       {'0.85':<15}")
print(f"{'Recall':<15} {test_metrics['recall']:.4f}       {'0.85':<15}")
print(f"{'F1 Score':<15} {test_metrics['f1']:.4f}       {'0.85':<15} ← Primary")
print(f"{'Accuracy':<15} {test_metrics['accuracy']:.4f}       {'0.99':<15}")

# Measure inference time (as per research paper Section IV-A)
print("INFERENCE TIME MEASUREMENT")

gcn_model.eval()
inference_times = []
WARMUP_SAMPLES = 20  # samples for warm-up
num_samples_to_measure = min(100, len(test_loader.dataset))

print(f"Warm-up samples: {WARMUP_SAMPLES}")
print(f"Measurement samples: {num_samples_to_measure}")

with torch.no_grad():
    samples_warmed = 0
    samples_measured = 0

    for batch1, batch2, perm_list in test_loader:
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)

        # Get number of graphs in this batch
        num_graphs = batch1.num_graphs if hasattr(batch1, 'num_graphs') else len(perm_list)

        for i in range(num_graphs):
            # Warm-up phase
            if samples_warmed < WARMUP_SAMPLES:
                S_pred_list, _ = gcn_model(batch1, batch2)
                _ = S_pred_list[i]  # Just to ensure computation
                samples_warmed += 1
                continue

            # Measurement phase
            if samples_measured >= num_samples_to_measure:
                break

            # Synchronize for accurate timing
            if device.type == 'cuda':
                torch.cuda.synchronize()

            start_time = time.perf_counter()
            S_pred_list, _ = gcn_model(batch1, batch2)
            _ = S_pred_list[i]  # Ensure computation is complete
            if device.type == 'cuda':
                torch.cuda.synchronize()

            inference_times.append(time.perf_counter() - start_time)
            samples_measured += 1

        if samples_measured >= num_samples_to_measure:
            break

avg_inference_time = np.mean(inference_times) * 1000  # ms
std_inference_time = np.std(inference_times) * 1000

print(f"\nAverage inference time: {avg_inference_time:.2f} ± {std_inference_time:.1f} ms")
print(f"Paper reported: 93 ms (0.093s)")
print(f"Speed relative to paper: {avg_inference_time/93:.2f}x")

# Distribution analysis
f1_scores = [m['f1'] for m in test_detailed]
print("\nF1 SCORE DISTRIBUTION ANALYSIS")
print(f"   Mean: {np.mean(f1_scores):.4f}")
print(f"   Median: {np.median(f1_scores):.4f}")
print(f"   Std: {np.std(f1_scores):.4f}")
print(f"   Min: {np.min(f1_scores):.4f}")
print(f"   Max: {np.max(f1_scores):.4f}")
print(f"   25th percentile: {np.percentile(f1_scores, 25):.4f}")
print(f"   75th percentile: {np.percentile(f1_scores, 75):.4f}")

# Plot test results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram of F1 scores
axes[0].hist(f1_scores, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(test_metrics['f1'], color='red', linestyle='--', linewidth=2,
                label=f'Mean: {test_metrics["f1"]:.3f}')
axes[0].axvline(np.median(f1_scores), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(f1_scores):.3f}')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of F1 Scores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(f1_scores, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_ylabel('F1 Score')
axes[1].set_title(f'F1 Score Distribution\nQ1: {np.percentile(f1_scores, 25):.3f}, '
                  f'Q2: {np.median(f1_scores):.3f}, Q3: {np.percentile(f1_scores, 75):.3f}')
axes[1].grid(True, alpha=0.3)

# Comparison bar chart
metrics_names = ['Precision', 'Recall', 'F1', 'Accuracy']
our_values = [test_metrics['precision'], test_metrics['recall'],
              test_metrics['f1'], test_metrics['accuracy']]
paper_values = [0.85, 0.85, 0.85, 0.99]

x = np.arange(len(metrics_names))
width = 0.35

axes[2].bar(x - width/2, our_values, width, label='GCN', color='steelblue')
axes[2].bar(x + width/2, paper_values, width, label='Paper Reported', color='lightcoral')
axes[2].set_xlabel('Metric')
axes[2].set_ylabel('Score')
axes[2].set_title('Comparison: GCN Results vs Research Paper')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].legend()
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GCN_test_results.png', dpi=300, bbox_inches='tight')
plt.show()

### Visualize Model Predictions on Test Set

In [ ]:
# Load original and noise graphs
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

with open(ORIGINAL_PATH, 'rb') as f:
    original_graphs_nx = pickle.load(f)
print(f"Loaded {len(original_graphs_nx)} original A-graphs")

with open(NOISE_PATH, 'rb') as f:
    noise_graphs_nx = pickle.load(f)
print(f"Loaded {len(noise_graphs_nx)} noise S-graphs")

# Get mean and std for normalization
mean, std = compute_mean_std(train_pairs)
print(f"Using precomputed mean and std for normalization")

def normalize_on_fly(g):
    """Apply normalization while preserving attributes."""
    g_norm = Data(x=(g.x - mean) / (std + 1e-8), edge_index=g.edge_index)
    if hasattr(g, 'name'):
        g_norm.name = g.name
    if hasattr(g, 'node_names'):
        g_norm.node_names = g.node_names
    if hasattr(g, 'permutation'):
        g_norm.permutation = g.permutation
    return g_norm

# Get test samples (using original test_pairs, not normalized)
num_samples = min(10, len(test_pairs))
print(f"\nVisualizing {num_samples} test samples with predictions...")

with torch.no_grad():
    for idx in range(num_samples):
        g1_orig, g2_orig, P_gt = test_pairs[idx]

        # Normalize on the fly for model input
        g1_norm = normalize_on_fly(g1_orig)
        g2_norm = normalize_on_fly(g2_orig)

        # Wrap as batches
        batch1 = Batch.from_data_list([g1_norm]).to(device)
        batch2 = Batch.from_data_list([g2_norm]).to(device)
        P_gt = P_gt.to(device)

        # Get predictions
        S_pred_list, _ = gcn_model(batch1, batch2)
        S_pred = S_pred_list[0]

        N1, N2 = P_gt.shape
        S_real = S_pred[:, :N2]

        # Hungarian assignment
        try:
            hard_assign = pygmtools.hungarian(
                S_real.unsqueeze(0),
                n1=torch.tensor([N1]),
                n2=torch.tensor([N2])
            ).squeeze(0)
        except:
            max_dim = max(N1, N2)
            S_padded = torch.zeros(max_dim, max_dim, device=S_real.device)
            S_padded[:N1, :N2] = S_real
            if N1 > N2:
                S_padded[N1:, :N2] = 1e-9
                S_padded[:N1, N2:] = 1e-9
            hard_assign_full = pygmtools.hungarian(S_padded.unsqueeze(0)).squeeze(0)
            hard_assign = hard_assign_full[:N1, :N2]

        # Evaluate all N1 × N2 pairs
        tp = fp = fn = tn = 0

        for i in range(N1):
            for j in range(N2):
                pred_match = (hard_assign[i, j] == 1)
                gt_match = (P_gt[i, j] == 1)

                if gt_match and pred_match:
                    tp += 1
                elif gt_match and not pred_match:
                    fn += 1
                elif not gt_match and pred_match:
                    fp += 1
                else:
                    tn += 1

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

        print(f"SAMPLE {idx+1}")
        print(f"  A-graph nodes: {g1_orig.x.shape[0]}")
        print(f"  S-graph nodes: {g2_orig.x.shape[0]}")
        print(f"  Ground truth matches: {P_gt.sum().item()}")
        print(f"  Total pairs evaluated: {N1 * N2}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

        # Print assignment details
        print(f"  Hungarian assignment matrix shape: {hard_assign.shape}")
        print(f"  Row sums (non-zero rows): {torch.where(hard_assign.sum(dim=1) > 0)[0].tolist()}")
        print(f"  Column sums: {hard_assign.sum(dim=0).tolist()}")

        # Check each ground truth match
        for i in range(N1):
            if (P_gt[i] > 0.5).any():
                gt_col = P_gt[i].argmax().item()
                pred_col = hard_assign[i].argmax().item() if hard_assign[i].sum() > 0 else -1
                print(f"    A-node {i}: GT→S-node {gt_col}, Pred→S-node {pred_col}, Correct: {pred_col == gt_col}")

        # Create visualization
        print(f"\n  Generating visualization...")
        fig, ax = plot_two_graphs_with_matching(
            graphs_list=[g1_orig, g2_orig],
            gt_perm=P_gt.cpu(),
            original_graphs=original_graphs_nx,
            pred_perm=hard_assign.cpu(),
            noise_graphs=noise_graphs_nx,
            viz_rooms=True,
            viz_ws=True,
            viz_room_connection=True,
            viz_normals=False,
            viz_room_normals=False,
            match_display="all",
            title=f"Test Sample {idx+1}: Predictions (F1={f1:.3f}) | ✓{tp} Correct, ✗{fp} Wrong",
            save_path=None
        )
        plt.show()
        plt.close(fig)

## GraphSAGE Encoder Model ([Hamilton et al., 2017](https://arxiv.org/abs/1706.02216))

### Train GraphSAGE Model

In [ ]:
# Load optimal hyperparameters
optimal_params = ModelParams.get_default()

print("\nOptimal Hyperparameters:")
for key, value in asdict(optimal_params).items():
    print(f"  {key}: {value}")

# Initialize model with optimal parameters
DEFAULT_PARAMS = ModelParams.get_default()

encoder = GraphSAGEEncoder(optimal_params)
graphSAGE_model = GraphMatcher(encoder, optimal_params).to(device)

# Break down parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
homogenizer_params = sum(p.numel() for p in encoder.homogenizer.parameters())
gat_params = sum(p.numel() for p in encoder.conv1.parameters()) + sum(p.numel() for p in encoder.conv2.parameters())

print(f"\nParameter Breakdown:")
print(f"  Encoder total: {encoder_params:,}")
print(f"    - MLP Homogenizer: {homogenizer_params:,}")
print(f"    - GraphSAGE layers: {gat_params:,}")
print(f"\nGraphSAGE Model Architecture:")
print(graphSAGE_model)

# Optimizer with optimal learning rate
optimizer = torch.optim.AdamW(
    graphSAGE_model.parameters(),
    lr=optimal_params.learning_rate,
    weight_decay=optimal_params.weight_decay
)

# Update dataloader with optimal batch size
BATCH_SIZE_OPTIMAL = optimal_params.batch_size

# Recreate dataloaders with optimal batch size
train_loader_optimal = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=True,
    collate_fn=collate_pyg_matching
)
val_loader_optimal = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)
test_loader_optimal = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)

print(f"\nDataLoaders recreated with batch_size={BATCH_SIZE_OPTIMAL}")

# Check for existing checkpoint to continue training
checkpoint_path = '/content/pretrained_models/best_GraphSAGE_model.pt'
start_epoch = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
val_f1_scores = []
val_precision = []
val_recall = []

if os.path.exists(checkpoint_path):
    print(f"\nFound existing checkpoint at {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        graphSAGE_model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        train_losses = checkpoint.get('train_losses', [])
        val_losses = checkpoint.get('val_losses', [])
        val_f1_scores = checkpoint.get('val_f1_scores', [])

        print(f"  Resuming from epoch {start_epoch}")
        print(f"  Previous best loss: {best_val_loss:.4f}")
    except Exception as e:
        print(f"  Could not load checkpoint: {e}")
        print(f"  Starting fresh training")
        start_epoch = 0
else:
    print(f"\nNo existing checkpoint found. Starting fresh training.")

NUM_EPOCHS = 45
PATIENCE = 5

patience_counter = 0

print(f"\nStarting training from epoch {start_epoch + 1}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    # Training Phase
    graphSAGE_model.train()
    epoch_train_loss = 0
    num_batches = 0

    for batch1, batch2, perm_list in tqdm(train_loader_optimal, desc=f"Epoch {epoch+1} Training"):
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)
        perm_list = [p.to(device) for p in perm_list]

        # Pass entire batches to model (returns list of S_pred for each graph in batch)
        S_pred_list, _ = graphSAGE_model(batch1, batch2)

        # Compute loss for each graph pair in the batch
        batch_loss = 0
        for i, S_pred in enumerate(S_pred_list):
            P_gt = perm_list[i]
            loss = permutation_loss(S_pred, P_gt)
            batch_loss += loss

        # Average loss over batch
        batch_loss = batch_loss / len(S_pred_list)

        # Backward pass
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        epoch_train_loss += batch_loss.item()
        num_batches += 1

    avg_train_loss = epoch_train_loss / num_batches if num_batches > 0 else 0
    train_losses.append(avg_train_loss)

    # Validation Phase
    val_metrics, val_detailed = evaluate(graphSAGE_model, val_loader_optimal, device)
    val_loss = val_metrics.get('loss', 0)
    val_losses.append(val_loss)
    val_f1_scores.append(val_metrics['f1'])
    val_precision.append(val_metrics['precision'])
    val_recall.append(val_metrics['recall'])

    current_lr = optimizer.param_groups[0]['lr']

    # Print progress every epoch
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val F1: {val_metrics['f1']:.4f} | "
          f"LR: {current_lr:.2e}")

    # Early Stopping (based on validation loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': graphSAGE_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_f1_scores': val_f1_scores,
            'hyperparams': optimal_params
        }, checkpoint_path)
        print(f"  → New best model! Val Loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best validation loss: {best_val_loss:.4f}")
            break

# Load best model
checkpoint = torch.load(checkpoint_path, weights_only=False)
graphSAGE_model.load_state_dict(checkpoint['model_state_dict'])

print("TRAINING COMPLETE")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(train_losses)}")

# Plot Training Curves
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot 1: Training and Validation Loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2, color='blue')
axes[0].plot(val_losses, label='Val Loss', linewidth=2, color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Validation F1 Score
axes[1].plot(val_f1_scores, label='Val F1 Score', linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle('Training Curves - GraphSAGE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GraphSAGE_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

### Test Set Evaluation

In [ ]:
# Load best model if not already loaded
if 'checkpoint' not in dir():
    checkpoint_path = '/content/pretrained_models/best_GraphSAGE_model.pt'
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        graphSAGE_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from {checkpoint_path}")
    else:
        print(f"Warning: No checkpoint found at {checkpoint_path}")
        print("Using current model state.")

# Evaluate on test set
print("TEST SET EVALUATION")

# Data is already preprocessed and normalized.
test_metrics, test_detailed = evaluate(graphSAGE_model, test_loader, device)

print("TEST SET RESULTS")
print(f"{'Metric':<15} {'Our Result':<15} {'Paper Reported':<15}")
print(f"{'Precision':<15} {test_metrics['precision']:.4f}       {'0.85':<15}")
print(f"{'Recall':<15} {test_metrics['recall']:.4f}       {'0.85':<15}")
print(f"{'F1 Score':<15} {test_metrics['f1']:.4f}       {'0.85':<15} ← Primary")
print(f"{'Accuracy':<15} {test_metrics['accuracy']:.4f}       {'0.99':<15}")

# Measure inference time (as per research paper Section IV-A)
print("INFERENCE TIME MEASUREMENT")

graphSAGE_model.eval()

inference_times = []
WARMUP_SAMPLES = 20  # samples for warm-up
num_samples_to_measure = min(100, len(test_loader.dataset))

print(f"Warm-up samples: {WARMUP_SAMPLES}")
print(f"Measurement samples: {num_samples_to_measure}")

with torch.no_grad():
    samples_warmed = 0
    samples_measured = 0

    for batch1, batch2, perm_list in test_loader:
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)

        # Get number of graphs in this batch
        num_graphs = batch1.num_graphs if hasattr(batch1, 'num_graphs') else len(perm_list)

        for i in range(num_graphs):
            # Warm-up phase
            if samples_warmed < WARMUP_SAMPLES:
                S_pred_list, _ = graphSAGE_model(batch1, batch2)
                _ = S_pred_list[i]  # Just to ensure computation
                samples_warmed += 1
                continue

            # Measurement phase
            if samples_measured >= num_samples_to_measure:
                break

            # Synchronize for accurate timing
            if device.type == 'cuda':
                torch.cuda.synchronize()

            start_time = time.perf_counter()
            S_pred_list, _ = graphSAGE_model(batch1, batch2)
            _ = S_pred_list[i]  # Ensure computation is complete
            if device.type == 'cuda':
                torch.cuda.synchronize()

            inference_times.append(time.perf_counter() - start_time)
            samples_measured += 1

        if samples_measured >= num_samples_to_measure:
            break

avg_inference_time = np.mean(inference_times) * 1000  # ms
std_inference_time = np.std(inference_times) * 1000

print(f"\nAverage inference time: {avg_inference_time:.2f} ± {std_inference_time:.1f} ms")
print(f"Paper reported: 93 ms (0.093s)")
print(f"Speed relative to paper: {avg_inference_time/93:.2f}x")

# Distribution analysis
f1_scores = [m['f1'] for m in test_detailed]
print("\nF1 SCORE DISTRIBUTION ANALYSIS")
print(f"   Mean: {np.mean(f1_scores):.4f}")
print(f"   Median: {np.median(f1_scores):.4f}")
print(f"   Std: {np.std(f1_scores):.4f}")
print(f"   Min: {np.min(f1_scores):.4f}")
print(f"   Max: {np.max(f1_scores):.4f}")
print(f"   25th percentile: {np.percentile(f1_scores, 25):.4f}")
print(f"   75th percentile: {np.percentile(f1_scores, 75):.4f}")

# Plot test results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram of F1 scores
axes[0].hist(f1_scores, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(test_metrics['f1'], color='red', linestyle='--', linewidth=2,
                label=f'Mean: {test_metrics["f1"]:.3f}')
axes[0].axvline(np.median(f1_scores), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(f1_scores):.3f}')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of F1 Scores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(f1_scores, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_ylabel('F1 Score')
axes[1].set_title(f'F1 Score Distribution\nQ1: {np.percentile(f1_scores, 25):.3f}, '
                  f'Q2: {np.median(f1_scores):.3f}, Q3: {np.percentile(f1_scores, 75):.3f}')
axes[1].grid(True, alpha=0.3)

# Comparison bar chart
metrics_names = ['Precision', 'Recall', 'F1', 'Accuracy']
our_values = [test_metrics['precision'], test_metrics['recall'],
              test_metrics['f1'], test_metrics['accuracy']]
paper_values = [0.85, 0.85, 0.85, 0.99]

x = np.arange(len(metrics_names))
width = 0.35

axes[2].bar(x - width/2, our_values, width, label='GraphSAGE', color='steelblue')
axes[2].bar(x + width/2, paper_values, width, label='Paper Reported', color='lightcoral')
axes[2].set_xlabel('Metric')
axes[2].set_ylabel('Score')
axes[2].set_title('Comparison: GraphSAGE Results vs Research Paper')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].legend()
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GraphSAGE_test_results.png', dpi=300, bbox_inches='tight')
plt.show()

### Visualize Model Predictions on Test Set

In [ ]:
# Load original and noise graphs
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

with open(ORIGINAL_PATH, 'rb') as f:
    original_graphs_nx = pickle.load(f)
print(f"Loaded {len(original_graphs_nx)} original A-graphs")

with open(NOISE_PATH, 'rb') as f:
    noise_graphs_nx = pickle.load(f)
print(f"Loaded {len(noise_graphs_nx)} noise S-graphs")

# Get mean and std for normalization
mean, std = compute_mean_std(train_pairs)
print(f"Using precomputed mean and std for normalization")

def normalize_on_fly(g):
    """Apply normalization while preserving attributes."""
    g_norm = Data(x=(g.x - mean) / (std + 1e-8), edge_index=g.edge_index)
    if hasattr(g, 'name'):
        g_norm.name = g.name
    if hasattr(g, 'node_names'):
        g_norm.node_names = g.node_names
    if hasattr(g, 'permutation'):
        g_norm.permutation = g.permutation
    return g_norm

# Get test samples (using original test_pairs, not normalized)
num_samples = min(10, len(test_pairs))
print(f"\nVisualizing {num_samples} test samples with predictions...")

with torch.no_grad():
    for idx in range(num_samples):
        g1_orig, g2_orig, P_gt = test_pairs[idx]

        # Normalize on the fly for model input
        g1_norm = normalize_on_fly(g1_orig)
        g2_norm = normalize_on_fly(g2_orig)

        # Wrap as batches
        batch1 = Batch.from_data_list([g1_norm]).to(device)
        batch2 = Batch.from_data_list([g2_norm]).to(device)
        P_gt = P_gt.to(device)

        # Get predictions
        S_pred_list, _ = graphSAGE_model(batch1, batch2)
        S_pred = S_pred_list[0]

        N1, N2 = P_gt.shape
        S_real = S_pred[:, :N2]

        # Hungarian assignment
        try:
            hard_assign = pygmtools.hungarian(
                S_real.unsqueeze(0),
                n1=torch.tensor([N1]),
                n2=torch.tensor([N2])
            ).squeeze(0)
        except:
            max_dim = max(N1, N2)
            S_padded = torch.zeros(max_dim, max_dim, device=S_real.device)
            S_padded[:N1, :N2] = S_real
            if N1 > N2:
                S_padded[N1:, :N2] = 1e-9
                S_padded[:N1, N2:] = 1e-9
            hard_assign_full = pygmtools.hungarian(S_padded.unsqueeze(0)).squeeze(0)
            hard_assign = hard_assign_full[:N1, :N2]

        # Evaluate all N1 × N2 pairs
        tp = fp = fn = tn = 0

        for i in range(N1):
            for j in range(N2):
                pred_match = (hard_assign[i, j] == 1)
                gt_match = (P_gt[i, j] == 1)

                if gt_match and pred_match:
                    tp += 1
                elif gt_match and not pred_match:
                    fn += 1
                elif not gt_match and pred_match:
                    fp += 1
                else:
                    tn += 1

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

        print(f"SAMPLE {idx+1}")
        print(f"  A-graph nodes: {g1_orig.x.shape[0]}")
        print(f"  S-graph nodes: {g2_orig.x.shape[0]}")
        print(f"  Ground truth matches: {P_gt.sum().item()}")
        print(f"  Total pairs evaluated: {N1 * N2}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

        # Print assignment details
        print(f"  Hungarian assignment matrix shape: {hard_assign.shape}")
        print(f"  Row sums (non-zero rows): {torch.where(hard_assign.sum(dim=1) > 0)[0].tolist()}")
        print(f"  Column sums: {hard_assign.sum(dim=0).tolist()}")

        # Check each ground truth match
        for i in range(N1):
            if (P_gt[i] > 0.5).any():
                gt_col = P_gt[i].argmax().item()
                pred_col = hard_assign[i].argmax().item() if hard_assign[i].sum() > 0 else -1
                print(f"    A-node {i}: GT→S-node {gt_col}, Pred→S-node {pred_col}, Correct: {pred_col == gt_col}")

        # Create visualization
        print(f"\n  Generating visualization...")
        fig, ax = plot_two_graphs_with_matching(
            graphs_list=[g1_orig, g2_orig],
            gt_perm=P_gt.cpu(),
            original_graphs=original_graphs_nx,
            pred_perm=hard_assign.cpu(),
            noise_graphs=noise_graphs_nx,
            viz_rooms=True,
            viz_ws=True,
            viz_room_connection=True,
            viz_normals=False,
            viz_room_normals=False,
            match_display="all",
            title=f"Test Sample {idx+1}: Predictions (F1={f1:.3f}) | ✓{tp} Correct, ✗{fp} Wrong",
            save_path=None
        )
        plt.show()
        plt.close(fig)

## GIN Encoder Model ([Xu et al., 2019](https://arxiv.org/abs/1810.00826))

### Train GIN Model

In [ ]:
# Load optimal hyperparameters
optimal_params = ModelParams.get_default()

print("\nOptimal Hyperparameters:")
for key, value in asdict(optimal_params).items():
    print(f"  {key}: {value}")

# Initialize model with optimal parameters
DEFAULT_PARAMS = ModelParams.get_default()

encoder = GINEncoder(optimal_params)
gin_model = GraphMatcher(encoder, optimal_params).to(device)

# Break down parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
homogenizer_params = sum(p.numel() for p in encoder.homogenizer.parameters())
gat_params = sum(p.numel() for p in encoder.conv1.parameters()) + sum(p.numel() for p in encoder.conv2.parameters())

print(f"\nParameter Breakdown:")
print(f"  Encoder total: {encoder_params:,}")
print(f"    - MLP Homogenizer: {homogenizer_params:,}")
print(f"    - GIN layers: {gat_params:,}")
print(f"\nGIN Model Architecture:")
print(gin_model)

# Optimizer with optimal learning rate
optimizer = torch.optim.AdamW(
    gin_model.parameters(),
    lr=optimal_params.learning_rate,
    weight_decay=optimal_params.weight_decay
)

# Update dataloader with optimal batch size
BATCH_SIZE_OPTIMAL = optimal_params.batch_size

# Recreate dataloaders with optimal batch size
train_loader_optimal = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=True,
    collate_fn=collate_pyg_matching
)
val_loader_optimal = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)
test_loader_optimal = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_OPTIMAL,
    shuffle=False,
    collate_fn=collate_pyg_matching
)

print(f"\nDataLoaders recreated with batch_size={BATCH_SIZE_OPTIMAL}")

# Check for existing checkpoint to continue training
checkpoint_path = '/content/pretrained_models/best_GIN_model.pt'
start_epoch = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
val_f1_scores = []
val_precision = []
val_recall = []

if os.path.exists(checkpoint_path):
    print(f"\nFound existing checkpoint at {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        gin_model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        train_losses = checkpoint.get('train_losses', [])
        val_losses = checkpoint.get('val_losses', [])
        val_f1_scores = checkpoint.get('val_f1_scores', [])

        print(f"  Resuming from epoch {start_epoch}")
        print(f"  Previous best loss: {best_val_loss:.4f}")
    except Exception as e:
        print(f"  Could not load checkpoint: {e}")
        print(f"  Starting fresh training")
        start_epoch = 0
else:
    print(f"\nNo existing checkpoint found. Starting fresh training.")

NUM_EPOCHS = 75
PATIENCE = 5

patience_counter = 0

print(f"\nStarting training from epoch {start_epoch + 1}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    # Training Phase
    gin_model.train()
    epoch_train_loss = 0
    num_batches = 0

    for batch1, batch2, perm_list in tqdm(train_loader_optimal, desc=f"Epoch {epoch+1} Training"):
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)
        perm_list = [p.to(device) for p in perm_list]

        # Pass entire batches to model (returns list of S_pred for each graph in batch)
        S_pred_list, _ = gin_model(batch1, batch2)

        # Compute loss for each graph pair in the batch
        batch_loss = 0
        for i, S_pred in enumerate(S_pred_list):
            P_gt = perm_list[i]
            loss = permutation_loss(S_pred, P_gt)
            batch_loss += loss

        # Average loss over batch
        batch_loss = batch_loss / len(S_pred_list)

        # Backward pass
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        epoch_train_loss += batch_loss.item()
        num_batches += 1

    avg_train_loss = epoch_train_loss / num_batches if num_batches > 0 else 0
    train_losses.append(avg_train_loss)

    # Validation Phase
    val_metrics, val_detailed = evaluate(gin_model, val_loader_optimal, device)
    val_loss = val_metrics.get('loss', 0)
    val_losses.append(val_loss)
    val_f1_scores.append(val_metrics['f1'])
    val_precision.append(val_metrics['precision'])
    val_recall.append(val_metrics['recall'])

    current_lr = optimizer.param_groups[0]['lr']

    # Print progress every epoch
    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val F1: {val_metrics['f1']:.4f} | "
          f"LR: {current_lr:.2e}")

    # Early Stopping (based on validation loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': gin_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_f1_scores': val_f1_scores,
            'hyperparams': optimal_params
        }, checkpoint_path)
        print(f"  → New best model! Val Loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best validation loss: {best_val_loss:.4f}")
            break

# Load best model
checkpoint = torch.load(checkpoint_path, weights_only=False)
gin_model.load_state_dict(checkpoint['model_state_dict'])

print("TRAINING COMPLETE")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(train_losses)}")

# Plot Training Curves
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot 1: Training and Validation Loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2, color='blue')
axes[0].plot(val_losses, label='Val Loss', linewidth=2, color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Validation F1 Score
axes[1].plot(val_f1_scores, label='Val F1 Score', linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle('Training Curves - GIN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GIN_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

### Test Set Evaluation

In [ ]:
# Load best model if not already loaded
if 'checkpoint' not in dir():
    checkpoint_path = '/content/pretrained_models/best_GIN_model.pt'
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        gin_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from {checkpoint_path}")
    else:
        print(f"Warning: No checkpoint found at {checkpoint_path}")
        print("Using current model state.")

# Evaluate on test set
print("TEST SET EVALUATION")

# Data is already preprocessed and normalized.
test_metrics, test_detailed = evaluate(gin_model, test_loader, device)

print("TEST SET RESULTS")
print(f"{'Metric':<15} {'Our Result':<15} {'Paper Reported':<15}")
print(f"{'Precision':<15} {test_metrics['precision']:.4f}       {'0.85':<15}")
print(f"{'Recall':<15} {test_metrics['recall']:.4f}       {'0.85':<15}")
print(f"{'F1 Score':<15} {test_metrics['f1']:.4f}       {'0.85':<15} ← Primary")
print(f"{'Accuracy':<15} {test_metrics['accuracy']:.4f}       {'0.99':<15}")

# Measure inference time (as per research paper Section IV-A)
print("INFERENCE TIME MEASUREMENT")

gin_model.eval()
inference_times = []
WARMUP_SAMPLES = 20  # samples for warm-up
num_samples_to_measure = min(100, len(test_loader.dataset))

print(f"Warm-up samples: {WARMUP_SAMPLES}")
print(f"Measurement samples: {num_samples_to_measure}")

with torch.no_grad():
    samples_warmed = 0
    samples_measured = 0

    for batch1, batch2, perm_list in test_loader:
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)

        # Get number of graphs in this batch
        num_graphs = batch1.num_graphs if hasattr(batch1, 'num_graphs') else len(perm_list)

        for i in range(num_graphs):
            # Warm-up phase
            if samples_warmed < WARMUP_SAMPLES:
                S_pred_list, _ = gin_model(batch1, batch2)
                _ = S_pred_list[i]  # Just to ensure computation
                samples_warmed += 1
                continue

            # Measurement phase
            if samples_measured >= num_samples_to_measure:
                break

            # Synchronize for accurate timing
            if device.type == 'cuda':
                torch.cuda.synchronize()

            start_time = time.perf_counter()
            S_pred_list, _ = gin_model(batch1, batch2)
            _ = S_pred_list[i]  # Ensure computation is complete
            if device.type == 'cuda':
                torch.cuda.synchronize()

            inference_times.append(time.perf_counter() - start_time)
            samples_measured += 1

        if samples_measured >= num_samples_to_measure:
            break

avg_inference_time = np.mean(inference_times) * 1000  # ms
std_inference_time = np.std(inference_times) * 1000

print(f"\nAverage inference time: {avg_inference_time:.2f} ± {std_inference_time:.1f} ms")
print(f"Paper reported: 93 ms (0.093s)")
print(f"Speed relative to paper: {avg_inference_time/93:.2f}x")

# Distribution analysis
f1_scores = [m['f1'] for m in test_detailed]
print("\nF1 SCORE DISTRIBUTION ANALYSIS")
print(f"   Mean: {np.mean(f1_scores):.4f}")
print(f"   Median: {np.median(f1_scores):.4f}")
print(f"   Std: {np.std(f1_scores):.4f}")
print(f"   Min: {np.min(f1_scores):.4f}")
print(f"   Max: {np.max(f1_scores):.4f}")
print(f"   25th percentile: {np.percentile(f1_scores, 25):.4f}")
print(f"   75th percentile: {np.percentile(f1_scores, 75):.4f}")

# Plot test results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram of F1 scores
axes[0].hist(f1_scores, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(test_metrics['f1'], color='red', linestyle='--', linewidth=2,
                label=f'Mean: {test_metrics["f1"]:.3f}')
axes[0].axvline(np.median(f1_scores), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(f1_scores):.3f}')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of F1 Scores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(f1_scores, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_ylabel('F1 Score')
axes[1].set_title(f'F1 Score Distribution\nQ1: {np.percentile(f1_scores, 25):.3f}, '
                  f'Q2: {np.median(f1_scores):.3f}, Q3: {np.percentile(f1_scores, 75):.3f}')
axes[1].grid(True, alpha=0.3)

# Comparison bar chart
metrics_names = ['Precision', 'Recall', 'F1', 'Accuracy']
our_values = [test_metrics['precision'], test_metrics['recall'],
              test_metrics['f1'], test_metrics['accuracy']]
paper_values = [0.85, 0.85, 0.85, 0.99]

x = np.arange(len(metrics_names))
width = 0.35

axes[2].bar(x - width/2, our_values, width, label='GIN', color='steelblue')
axes[2].bar(x + width/2, paper_values, width, label='Paper Reported', color='lightcoral')
axes[2].set_xlabel('Metric')
axes[2].set_ylabel('Score')
axes[2].set_title('Comparison: GIN Results vs Research Paper')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].legend()
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/GIN_test_results.png', dpi=300, bbox_inches='tight')
plt.show()

### Visualize Model Predictions on Test Set

In [ ]:
# Load original and noise graphs
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

with open(ORIGINAL_PATH, 'rb') as f:
    original_graphs_nx = pickle.load(f)
print(f"Loaded {len(original_graphs_nx)} original A-graphs")

with open(NOISE_PATH, 'rb') as f:
    noise_graphs_nx = pickle.load(f)
print(f"Loaded {len(noise_graphs_nx)} noise S-graphs")

# Get mean and std for normalization
mean, std = compute_mean_std(train_pairs)
print(f"Using precomputed mean and std for normalization")

def normalize_on_fly(g):
    """Apply normalization while preserving attributes."""
    g_norm = Data(x=(g.x - mean) / (std + 1e-8), edge_index=g.edge_index)
    if hasattr(g, 'name'):
        g_norm.name = g.name
    if hasattr(g, 'node_names'):
        g_norm.node_names = g.node_names
    if hasattr(g, 'permutation'):
        g_norm.permutation = g.permutation
    return g_norm

# Get test samples (using original test_pairs, not normalized)
num_samples = min(10, len(test_pairs))
print(f"\nVisualizing {num_samples} test samples with predictions...")

with torch.no_grad():
    for idx in range(num_samples):
        g1_orig, g2_orig, P_gt = test_pairs[idx]

        # Normalize on the fly for model input
        g1_norm = normalize_on_fly(g1_orig)
        g2_norm = normalize_on_fly(g2_orig)

        # Wrap as batches
        batch1 = Batch.from_data_list([g1_norm]).to(device)
        batch2 = Batch.from_data_list([g2_norm]).to(device)
        P_gt = P_gt.to(device)

        # Get predictions
        S_pred_list, _ = gin_model(batch1, batch2)
        S_pred = S_pred_list[0]

        N1, N2 = P_gt.shape
        S_real = S_pred[:, :N2]

        # Hungarian assignment
        try:
            hard_assign = pygmtools.hungarian(
                S_real.unsqueeze(0),
                n1=torch.tensor([N1]),
                n2=torch.tensor([N2])
            ).squeeze(0)
        except:
            max_dim = max(N1, N2)
            S_padded = torch.zeros(max_dim, max_dim, device=S_real.device)
            S_padded[:N1, :N2] = S_real
            if N1 > N2:
                S_padded[N1:, :N2] = 1e-9
                S_padded[:N1, N2:] = 1e-9
            hard_assign_full = pygmtools.hungarian(S_padded.unsqueeze(0)).squeeze(0)
            hard_assign = hard_assign_full[:N1, :N2]

        # Evaluate all N1 × N2 pairs
        tp = fp = fn = tn = 0

        for i in range(N1):
            for j in range(N2):
                pred_match = (hard_assign[i, j] == 1)
                gt_match = (P_gt[i, j] == 1)

                if gt_match and pred_match:
                    tp += 1
                elif gt_match and not pred_match:
                    fn += 1
                elif not gt_match and pred_match:
                    fp += 1
                else:
                    tn += 1

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)

        print(f"SAMPLE {idx+1}")
        print(f"  A-graph nodes: {g1_orig.x.shape[0]}")
        print(f"  S-graph nodes: {g2_orig.x.shape[0]}")
        print(f"  Ground truth matches: {P_gt.sum().item()}")
        print(f"  Total pairs evaluated: {N1 * N2}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

        # Print assignment details
        print(f"  Hungarian assignment matrix shape: {hard_assign.shape}")
        print(f"  Row sums (non-zero rows): {torch.where(hard_assign.sum(dim=1) > 0)[0].tolist()}")
        print(f"  Column sums: {hard_assign.sum(dim=0).tolist()}")

        # Check each ground truth match
        for i in range(N1):
            if (P_gt[i] > 0.5).any():
                gt_col = P_gt[i].argmax().item()
                pred_col = hard_assign[i].argmax().item() if hard_assign[i].sum() > 0 else -1
                print(f"    A-node {i}: GT→S-node {gt_col}, Pred→S-node {pred_col}, Correct: {pred_col == gt_col}")

        # Create visualization
        print(f"\n  Generating visualization...")
        fig, ax = plot_two_graphs_with_matching(
            graphs_list=[g1_orig, g2_orig],
            gt_perm=P_gt.cpu(),
            original_graphs=original_graphs_nx,
            pred_perm=hard_assign.cpu(),
            noise_graphs=noise_graphs_nx,
            viz_rooms=True,
            viz_ws=True,
            viz_room_connection=True,
            viz_normals=False,
            viz_room_normals=False,
            match_display="all",
            title=f"Test Sample {idx+1}: Predictions (F1={f1:.3f}) | ✓{tp} Correct, ✗{fp} Wrong",
            save_path=None
        )
        plt.show()
        plt.close(fig)

## Aggregated GNN Encoder Model Comparison

In [ ]:
# Encoder mapping
ENCODERS = {
    'GCN': GCNEncoder,
    'GraphSAGE': GraphSAGEEncoder,
    'GIN': GINEncoder,
    'GraphTransformer': GraphTransformerEncoder,
    'GATv2': GATv2Encoder
}

# Initialize models with optimal parameters
DEFAULT_PARAMS = ModelParams.get_default()

# Checkpoint paths for each model
CHECKPOINT_PATHS = {
    'GCN': '/content/pretrained_models/best_GCN_model.pt',
    'GraphSAGE': '/content/pretrained_models/best_GraphSAGE_model.pt',
    'GIN': '/content/pretrained_models/best_GIN_model.pt',
    'GraphTransformer': '/content/pretrained_models/best_GraphTransformer_model.pt',
    'GATv2': '/content/pretrained_models/best_GATv2_model.pt'
}

encoder_results = {}
encoder_models = {}

def evaluate_inference_time(model, test_loader, device, num_samples=100):
    """Measure average inference time."""
    model.eval()
    inference_times = []

    with torch.no_grad():
        samples_collected = 0
        for batch1, batch2, perm_list in test_loader:
            if samples_collected >= num_samples:
                break

            # Move entire batch to device
            batch1 = batch1.to(device)
            batch2 = batch2.to(device)

            # Warm-up (20 runs)
            for _ in range(20):
                _, _ = model(batch1, batch2)

            # Measure
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.perf_counter()
            _, _ = model(batch1, batch2)
            if device.type == 'cuda':
                torch.cuda.synchronize()

            # Count samples in this batch
            num_graphs = batch1.num_graphs if hasattr(batch1, 'num_graphs') else len(perm_list)
            inference_times.extend([time.perf_counter() - start_time] * num_graphs)
            samples_collected += num_graphs

    avg_time = np.mean(inference_times[:num_samples]) * 1000
    std_time = np.std(inference_times[:num_samples]) * 1000
    return avg_time, std_time

def load_pretrained_model_with_history(encoder_name, EncoderClass, params, device, checkpoint_path):
    """Load a pretrained model and its training history if checkpoint exists."""
    if os.path.exists(checkpoint_path):
        print(f"  Loading pretrained model from {checkpoint_path}")
        encoder = EncoderClass(params)
        model = GraphMatcher(encoder, params).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        # Load training history
        training_history = {
            'train_losses': checkpoint.get('train_losses', []),
            'val_losses': checkpoint.get('val_losses', []),
            'val_f1_scores': checkpoint.get('val_f1_scores', []),
            'best_val_loss': checkpoint.get('best_val_loss', float('inf')),
            'best_val_f1': checkpoint.get('best_val_f1', 0),
            'epoch': checkpoint.get('epoch', 0)
        }

        # Count parameters
        num_params = sum(p.numel() for p in model.parameters())
        print(f"  Total parameters: {num_params:,}")
        print(f"  Loaded from epoch {training_history['epoch']+1}")
        print(f"  Best validation loss: {training_history['best_val_loss']:.4f}")

        return model, num_params, training_history
    else:
        print(f"  No pretrained model found at {checkpoint_path}")
        return None, None, None

def evaluate_model(model, test_loader, device):
    """Evaluate a single model and return metrics."""
    model.eval()
    test_metrics, test_detailed = evaluate(model, test_loader, device)
    return test_metrics, test_detailed

print("PRETRAINED GNN MODEL COMPARISON")

# Load and evaluate all models from pretrained checkpoints
for encoder_name, EncoderClass in ENCODERS.items():
    print(f"Processing {encoder_name}...")

    params = DEFAULT_PARAMS
    checkpoint_path = CHECKPOINT_PATHS[encoder_name]

    # Load pretrained model with history
    trained_model, num_params, training_history = load_pretrained_model_with_history(
        encoder_name, EncoderClass, params, device, checkpoint_path
    )

    if trained_model is None:
        print(f"  Skipping {encoder_name} - no pretrained model found")
        continue

    # Evaluate on test set
    print(f"\n  Evaluating {encoder_name} on test set...")
    test_metrics, test_detailed = evaluate_model(trained_model, test_loader, device)

    # Measure inference time
    inference_time_mean, inference_time_std = evaluate_inference_time(
        trained_model, test_loader, device, num_samples=200
    )

    # Store results
    encoder_results[encoder_name] = {
        'test_f1': test_metrics['f1'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_accuracy': test_metrics['accuracy'],
        'inference_time_ms': inference_time_mean,
        'inference_time_std': inference_time_std,
        'num_params': num_params,
        'training_history': training_history
    }

    encoder_models[encoder_name] = trained_model

    print(f"\n  {encoder_name} Results Summary:")
    print(f"    Test F1: {test_metrics['f1']:.4f} (Primary)")
    print(f"    Test Precision: {test_metrics['precision']:.4f}")
    print(f"    Test Recall: {test_metrics['recall']:.4f}")
    print(f"    Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"    Inference Time: {inference_time_mean:.2f} ± {inference_time_std:.1f} ms")
    print(f"    Parameters: {num_params:,}")

# Check if any models were loaded
if len(encoder_results) == 0:
    print("WARNING: No pretrained models found!")
    print("Please train the models first or check checkpoint paths.")
else:

    # Create figure with multiple subplots
    fig = plt.figure(figsize=(18, 12))

    # Color map for different models
    colors = {
        'GCN': 'blue',
        'GraphSAGE': 'orange',
        'GIN': 'green',
        'GraphTransformer': 'purple',
        'GATv2': 'red'
    }

    markers = {
        'GCN': 'o',
        'GraphSAGE': 's',
        'GIN': '^',
        'GraphTransformer': 'D',
        'GATv2': '*'
    }

    # Plot 1: Training Loss Curves
    ax1 = fig.add_subplot(2, 3, 1)
    for encoder_name, results in encoder_results.items():
        train_losses = results['training_history']['train_losses']
        if len(train_losses) > 0:
            epochs = range(1, len(train_losses) + 1)
            ax1.plot(epochs, train_losses,
                     label=encoder_name,
                     color=colors[encoder_name],
                     linewidth=2,
                     marker=markers[encoder_name],
                     markevery=max(1, len(train_losses)//10))
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss Curves by Model')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')

    # Plot 2: Validation Loss Curves
    ax2 = fig.add_subplot(2, 3, 2)
    for encoder_name, results in encoder_results.items():
        val_losses = results['training_history']['val_losses']
        if len(val_losses) > 0:
            epochs = range(1, len(val_losses) + 1)
            ax2.plot(epochs, val_losses,
                     label=encoder_name,
                     color=colors[encoder_name],
                     linewidth=2,
                     marker=markers[encoder_name],
                     markevery=max(1, len(val_losses)//10))
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Loss')
    ax2.set_title('Validation Loss Curves by Model')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    ax2.set_yscale('log')

    # Plot 3: Validation F1 Curves
    ax3 = fig.add_subplot(2, 3, 3)
    for encoder_name, results in encoder_results.items():
        val_f1 = results['training_history']['val_f1_scores']
        if len(val_f1) > 0:
            epochs = range(1, len(val_f1) + 1)
            ax3.plot(epochs, val_f1,
                     label=encoder_name,
                     color=colors[encoder_name],
                     linewidth=2,
                     marker=markers[encoder_name],
                     markevery=max(1, len(val_f1)//10))
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Validation F1 Score')
    ax3.set_title('Validation F1 Curves by Model')
    ax3.legend(loc='lower right')
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)

    # Plot 4: Parameter Count Comparison
    ax4 = fig.add_subplot(2, 3, 4)

    encoder_names = list(encoder_results.keys())
    param_counts = [encoder_results[a]['num_params'] for a in encoder_names]
    bar_colors = [colors[e] for e in encoder_names]

    bars = ax4.bar(encoder_names, param_counts, color=bar_colors, edgecolor='black', linewidth=1.5)

    ax4.set_ylabel('Number of Parameters')
    ax4.set_title('Model Size by Encoder Name')
    ax4.set_yscale('log')

    # Annotate bars
    for bar, count in zip(bars, param_counts):
        ax4.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() * 1.05,
            f"{count:,}",
            ha='center',
            fontsize=9,
            fontweight='bold'
        )

    ax4.grid(True, alpha=0.3, axis='y')
#arch
    # Plot 5: Test F1 Comparison Bar Chart
    ax5 = fig.add_subplot(2, 3, 5)
    encoder_names = list(encoder_results.keys())
    test_f1_scores = [encoder_results[a]['test_f1'] for a in encoder_names]
    bar_colors = [colors[a] for a in encoder_names]
    bars = ax5.bar(encoder_names, test_f1_scores, color=bar_colors, edgecolor='black', linewidth=1.5)
    ax5.set_ylabel('F1 Score')
    ax5.set_title('Test F1 Score by Model')
    ax5.set_ylim(0, 1)
    for bar, score in zip(bars, test_f1_scores):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{score:.4f}', ha='center', fontsize=10, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='y')

    # Plot 6: Inference Time Comparison
    ax6 = fig.add_subplot(2, 3, 6)
    inference_times = [encoder_results[a]['inference_time_ms'] for a in encoder_names]
    bars = ax6.bar(encoder_names, inference_times, color=bar_colors, edgecolor='black', linewidth=1.5)
    ax6.set_ylabel('Inference Time (ms)')
    ax6.set_title('Inference Time by Model')
    ax6.set_yscale('log')
    for bar, time_ms in zip(bars, inference_times):
        ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{time_ms:.1f}ms', ha='center', fontsize=9)
    ax6.grid(True, alpha=0.3, axis='y')

    plt.suptitle('Pretrained GNN Model Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/results/pretrained_GNN_model_comparison_pretrained.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Identify best model
    best_encoder = max(encoder_results.keys(), key=lambda x: encoder_results[x]['test_f1'])
    best_f1 = encoder_results[best_encoder]['test_f1']

    print(f"BEST PERFORMING MODEL: {best_encoder}")
    print(f"   Test F1: {best_f1:.4f}")

    # Statistical comparison
    if 'GATv2' in encoder_results:
        print("\nSTATISTICAL COMPARISON vs GATv2 (Paper's proposed)")
        gatv2_f1 = encoder_results['GATv2']['test_f1']
        print(f"\nGATv2 F1: {gatv2_f1:.4f}")
        print("\nPerformance difference vs other models:")
        for encoder in encoder_results.keys():
            if encoder != 'GATv2':
                diff = (encoder_results[encoder]['test_f1'] - gatv2_f1) * 100
                symbol = "↑" if diff > 0 else "↓"
                print(f"  {encoder:10s}: {symbol} {abs(diff):+.2f}% F1 difference")

    # Save comparison table
    comparison_df = pd.DataFrame([
        {
            'Model': encoder,
            'Test F1': f"{results['test_f1']:.4f}",
            'Test Precision': f"{results['test_precision']:.4f}",
            'Test Recall': f"{results['test_recall']:.4f}",
            'Inference Time (ms)': f"{results['inference_time_ms']:.2f}",
            'Parameters': f"{results['num_params']:,}",
            'Best Val Loss': f"{results['training_history'].get('best_val_loss', 0):.4f}"
        }
        for encoder, results in encoder_results.items()
    ])

    print("DETAILED COMPARISON TABLE")
    print(comparison_df.to_string(index=False))

    # Save to CSV
    comparison_df.to_csv('/content/results/pretrained_GNN_model_comparison.csv', index=False)

## **What is GNNExplainer?**

GNNExplainer is a **model‑agnostic** method that explains the predictions of any Graph Neural Network (GNN). Given a trained GNN and a specific prediction task, GNNExplainer identifies a **compact subgraph** of the input graph, and a **small subset of node features** that are most influential for that prediction. GNNExplainer works for node classification, link prediction, and graph classification without modifying the GNN architecture.

**Paper:** [GNNExplainer: Generating Explanations for Graph Neural Networks](https://arxiv.org/abs/1903.03894) (Ying et al., 2019)

### **Mathematical Formulation** (from the paper)

#### 1. Mutual Information Objective (Core Explanation Goal)

For a node $v$, GNNExplainer finds a subgraph $G_S$ (with associated features $X_S$) that maximizes the mutual information with the GNN’s prediction $Y$:

$$
\max_{G_S} \text{MI}(Y, (G_S, X_S)) = H(Y) - H(Y \mid G = G_S, X = X_S)
$$

*Source: Equation (1), Section 4.1*

- $H(Y)$ = entropy of the prediction (uncertainty before seeing the subgraph) – **constant** for a trained GNN.  
- $H(Y | G = G_S, X = X_S)$ = conditional entropy (remaining uncertainty after seeing $G_S$).

Because $H(Y)$ is fixed, maximizing mutual information is **equivalent to minimizing the conditional entropy** $H(Y \mid G_S, X_S)$.


#### 2. Conditional Entropy Formulation

The conditional entropy is written as:

$$
H(Y \mid G = G_S, X = X_S)
= -\mathbb{E}_{Y \mid G_S, X_S}\bigl[\log P_{\Phi}(Y \mid G = G_S, X = X_S)\bigr]
$$

*Source: Equation (2), Section 4.1*

Where $P_{\Phi}$ is is the probability distribution learned by the GNN ${\Phi}$

#### 3. Variational Mask Approximation

Directly searching over all subgraphs is intractable. GNNExplainer therefore uses a **mean‑field variational approximation**, replacing the discrete subgraph $G_S$ with its expectation under a factorized distribution:

$$
\min_{\mathcal{G}} \; H(Y \mid G = \mathbb{E}_{\mathcal{G}}[G_S], X = X_S)
$$

*Source: Equation (4), Section 4.1*

The expectation is computed with a **mask matrix** $M \in \mathbb{R}^{n \times n}$:

$$
\mathbb{E}_{\mathcal{G}}[G_S] = A_c \odot \sigma(M)
$$

- $A_c$ = adjacency matrix of the **computation graph**  
- $\odot$ = element‑wise multiplication  
- $\sigma$ = sigmoid function mapping mask values to $[0,1]$

This produces a **soft adjacency matrix**, where each entry represents the **probability that an edge is included** in the explanation subgraph.

#### 4. Cross‑Entropy Objective for Label‑Specific Explanations

In scenarios where users care about why the model predicts a specific class or how to make the model predict a desired class, GNNExplainer replaces the mutual information objective with a cross‑entropy objective that directly aligns the explanation with a target label.

The optimization problem becomes:

$$
\min_{M} \; -\sum_{c=1}^{C} \mathbf{1}[y=c] \log P_{\Phi}(Y=y \mid G = A_c \odot \sigma(M), X = X_c)
$$

*Source: Equation (5), Section 4.1*

This objective computes the **negative log‑likelihood** of the desired label
$y$ under the masked graph, enabling efficient gradient‑based optimization of the mask
$M$.


### **Our Objective: Node Classification for Graph Matching**

We trained a node classifier using our pretrained GNN model to predict, for each node $x$ in the S‑graph $S$, which nodes and edges in the A‑graph contributed to the node prediction.

The GNN is frozen during this process and only the classifier head is trained.

The classifier computes:

$$
\text{logits} = \text{NodeClassifier}(\text{GNN}(x, S))
$$

and is optimized using:

$$
\text{loss} = \text{Cross-Entropy}(\text{logits}, \text{labels})
$$

The classifier learns to map encoder embeddings to match **labels** obtained from the ground‑truth permutation matrix:

$$
\text{labels} = \arg\max_{j}\,(P_{\text{gt}})_{i,j}
$$


#### **What GNNExplainer Tells Us**

For a **given S‑graph node**, GNNExplainer identifies which A‑graph nodes and edges were most influential in the model’s prediction.

#### Visualization Legend

| Element | Meaning |
|--------|---------|
| **Normalized Importance (colorbar)** | Importance to prediction (High 🔴 →🔵 Low) |
| **Gold dot** | Target S‑node being explained |
| **Green line** | Correct match prediction |
| **Red line** | Wrong match prediction |
| **Orange dashed line** | Missing ground‑truth match |

#### Code Implementation (PyG Documentation)

```python
explainer = Explainer(
    model=classifier,
    algorithm=GNNExplainer(epochs=100, lr=0.01),
    explanation_type='model',
    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='raw',
    ),
    node_mask_type='attributes',  # learns which node feature dimensions matter
    edge_mask_type='object',      # learns which edges matter
)
```


### References

1. **Original Paper:** Ying et al., *GNNExplainer: Generating Explanations for Graph Neural Networks*, NeurIPS 2019.  
   [https://arxiv.org/abs/1903.03894](https://arxiv.org/abs/1903.03894)  
2. **PyG Documentation:**  
   [https://pytorch-geometric.readthedocs.io/en/latest/modules/explain.html](https://pytorch-geometric.readthedocs.io/en/latest/modules/explain.html)

## GNNExplainer Implementation

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer

### Model Explainability Utility Functions

In [ ]:
# Node Classifier Wrapper
class NodeClassifier(nn.Module):
    """Node classifier that uses the pretrained encoder"""
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.output_dim, num_classes)

    def forward(self, x, edge_index):
        data = Data(x=x, edge_index=edge_index)
        h = self.encoder(data)
        return self.classifier(h)


def get_misclassified_node_and_predictions(model, g1_orig, g2_orig, P_gt, device, mean, std):
    """
    Get hard assignments and find a misclassified A-node in one pass.
    """
    def normalize(g):
        return Data(x=(g.x - mean) / (std + 1e-8), edge_index=g.edge_index)

    g1_norm = normalize(g1_orig).to(device)
    g2_norm = normalize(g2_orig).to(device)

    model.eval()
    with torch.no_grad():
        batch1 = Batch.from_data_list([g1_norm])
        batch2 = Batch.from_data_list([g2_norm])
        S_pred_list, _ = model(batch1, batch2)
        S_pred = S_pred_list[0]

        N1, N2 = P_gt.shape
        S_real = S_pred[:, :N2]
        hard_assign = pygmtools.hungarian(S_real.unsqueeze(0)).squeeze(0)
        pred_labels = hard_assign.argmax(dim=1)
        true_labels = P_gt.argmax(dim=1)

    # Find misclassified A-nodes
    misclassified_nodes = []
    for j in range(N1):
        if P_gt[j].sum().item() > 0:
            if pred_labels[j].item() != true_labels[j].item():
                misclassified_nodes.append(j)

    return hard_assign.cpu(), pred_labels.cpu(), true_labels.cpu(), misclassified_nodes


def get_gnnexplainer_importance_for_misclassified(model, g1_orig, g2_orig, P_gt, device, mean, std, epochs=100):
    """
    Use GNNExplainer on a misclassified node.
    """
    hard_assign, pred_labels, true_labels, misclassified_nodes = get_misclassified_node_and_predictions(
        model, g1_orig, g2_orig, P_gt, device, mean, std
    )

    if misclassified_nodes:
        node_idx = misclassified_nodes[0]
        pred_class = pred_labels[node_idx].item()
        true_class = true_labels[node_idx].item()
        print(f"  Found misclassified A-node: {node_idx} (Pred: S-{pred_class}, True: S-{true_class})")
    else:
        node_idx = 0
        pred_class = pred_labels[node_idx].item()
        true_class = true_labels[node_idx].item()
        print(f"  WARNING: No misclassified node found, using node {node_idx}")

    print(f"    Explaining node {node_idx} (Pred: S-{pred_class}, True: S-{true_class})")

    # Freeze encoder
    for param in model.encoder.parameters():
        param.requires_grad = False

    classifier = NodeClassifier(model.encoder, P_gt.shape[1]).to(device)

    # Train classifier on this sample
    x = g1_orig.x.to(device)
    edge_index = g1_orig.edge_index.to(device)
    labels = P_gt.argmax(dim=1).to(device)

    optimizer = torch.optim.Adam(classifier.classifier.parameters(), lr=0.01)
    for epoch in range(epochs):
        classifier.train()
        optimizer.zero_grad()
        out = classifier(x, edge_index)
        loss = F.cross_entropy(out, labels)
        loss.backward()
        optimizer.step()

    classifier.eval()
    with torch.no_grad():
        logits = classifier(x, edge_index)
        accuracy = (logits.argmax(dim=1) == labels).float().mean().item()

    # Create explainer
    explainer = Explainer(
        model=classifier,
        algorithm=GNNExplainer(epochs=100, lr=0.01),
        explanation_type='model',
        model_config=dict(
            mode='multiclass_classification',
            task_level='node',
            return_type='raw',
        ),
        node_mask_type='attributes',
        edge_mask_type='object',
    )

    explanation = explainer(x=x, edge_index=edge_index, index=node_idx)

    # Extract node mask (importance per node, possibly per feature)
    node_mask = explanation.node_mask.cpu().numpy()

    # Extract feature importance for the target node (if 2D mask)
    if node_mask.ndim == 2:
        feature_importance = node_mask[node_idx]  # Shape: [num_features]
        node_importance = node_mask.mean(axis=1)  # Average across features
    else:
        feature_importance = None
        node_importance = node_mask

    # Extract edge importance
    edge_importance = explanation.edge_mask.cpu().numpy()
    if edge_importance.ndim > 1:
        edge_importance = edge_importance.mean(axis=1)

    # Print feature importance
    feature_names = ['Type_Room', 'Type_WS', 'Centroid_X', 'Centroid_Y', 'Normal_X', 'Normal_Y', 'Segment_Length']
    if feature_importance is not None:
        print(f"    Feature importance for node {node_idx}:")
        for f_idx, (f_name, imp) in enumerate(zip(feature_names, feature_importance)):
            print(f"      {f_name:15s}: {imp:.4f}")

    print(f"    Node importance range: [{node_importance.min():.4f}, {node_importance.max():.4f}]")
    print(f"    Edge importance range: [{edge_importance.min():.4f}, {edge_importance.max():.4f}]")

    return node_importance, edge_importance, feature_importance, node_idx, pred_class, true_class, accuracy, hard_assign


def get_gnnexplainer_importance_for_correct(model, g1_orig, g2_orig, P_gt, device, mean, std, epochs=50):
    """
    Use GNNExplainer on a correctly classified node.
    """
    hard_assign, pred_labels, true_labels, _ = get_misclassified_node_and_predictions(
        model, g1_orig, g2_orig, P_gt, device, mean, std
    )

    # Find a correctly classified node
    node_idx = 0
    for j in range(P_gt.shape[0]):
        if P_gt[j].sum().item() > 0:
            if pred_labels[j].item() == true_labels[j].item():
                node_idx = j
                break

    pred_class = pred_labels[node_idx].item()
    true_class = true_labels[node_idx].item()
    print(f"  Using correctly classified A-node: {node_idx} (Pred: S-{pred_class}, True: S-{true_class})")
    print(f"    Explaining node {node_idx}")

    # Freeze encoder
    for param in model.encoder.parameters():
        param.requires_grad = False

    classifier = NodeClassifier(model.encoder, P_gt.shape[1]).to(device)

    # Train classifier
    x = g1_orig.x.to(device)
    edge_index = g1_orig.edge_index.to(device)
    labels = P_gt.argmax(dim=1).to(device)

    optimizer = torch.optim.Adam(classifier.classifier.parameters(), lr=0.01)
    for epoch in range(epochs):
        classifier.train()
        optimizer.zero_grad()
        out = classifier(x, edge_index)
        loss = F.cross_entropy(out, labels)
        loss.backward()
        optimizer.step()

    classifier.eval()
    with torch.no_grad():
        logits = classifier(x, edge_index)
        accuracy = (logits.argmax(dim=1) == labels).float().mean().item()

    # Create explainer
    explainer = Explainer(
        model=classifier,
        algorithm=GNNExplainer(epochs=100, lr=0.01),
        explanation_type='model',
        model_config=dict(
            mode='multiclass_classification',
            task_level='node',
            return_type='raw',
        ),
        node_mask_type='attributes',
        edge_mask_type='object',
    )

    explanation = explainer(x=x, edge_index=edge_index, index=node_idx)

    # Extract node mask
    node_mask = explanation.node_mask.cpu().numpy()

    # Extract feature importance for the target node
    if node_mask.ndim == 2:
        feature_importance = node_mask[node_idx]
        node_importance = node_mask.mean(axis=1)
    else:
        feature_importance = None
        node_importance = node_mask

    # Extract edge importance
    edge_importance = explanation.edge_mask.cpu().numpy()
    if edge_importance.ndim > 1:
        edge_importance = edge_importance.mean(axis=1)

    # Print feature importance
    feature_names = ['Type_Room', 'Type_WS', 'Centroid_X', 'Centroid_Y', 'Normal_X', 'Normal_Y', 'Segment_Length']
    if feature_importance is not None:
        print(f"    Feature importance for node {node_idx}:")
        for f_idx, (f_name, imp) in enumerate(zip(feature_names, feature_importance)):
            print(f"      {f_name:15s}: {imp:.4f}")

    print(f"    Node importance range: [{node_importance.min():.4f}, {node_importance.max():.4f}]")
    print(f"    Edge importance range: [{edge_importance.min():.4f}, {edge_importance.max():.4f}]")

    return node_importance, edge_importance, feature_importance, node_idx, pred_class, true_class, accuracy, hard_assign

def get_matching_predictions(model, g1_orig, g2_orig, device, mean, std):
    """Get hard assignments using the pretrained model."""

    def normalize(g):
        return Data(x=(g.x - mean) / (std + 1e-8), edge_index=g.edge_index)

    g1_norm = normalize(g1_orig).to(device)
    g2_norm = normalize(g2_orig).to(device)

    model.eval()

    with torch.no_grad():
        batch1 = Batch.from_data_list([g1_norm])
        batch2 = Batch.from_data_list([g2_norm])
        S_pred_list, _ = model(batch1, batch2)
        S_pred = S_pred_list[0]
        hard_assign = pygmtools.hungarian(S_pred.unsqueeze(0)).squeeze(0)

    return hard_assign.cpu()

def plot_node_importance_with_graphs(graphs_list, gt_perm, original_graphs,
                                      node_importance, target_node_id,
                                      pred_perm=None, noise_graphs=None,
                                      title=None, save_path=None):
    """
    Node importance visualization with both A-graph and S-graph side by side.
    """

    if noise_graphs is None:
        noise_graphs = original_graphs

    g1tensor, g2tensor = copy.deepcopy(graphs_list[0]), copy.deepcopy(graphs_list[1])

    # Check if S-graph has nodes
    if g2tensor.x.shape[0] == 0:
        print(f"  WARNING: S-graph has no nodes, skipping visualization")
        return None, None

    node_names1 = list(g1tensor.node_names)
    orig_names2 = list(g2tensor.node_names)
    perm = g2tensor.permutation.tolist()
    node_names2 = [orig_names2[p] for p in perm]

    # Convert to NetworkX
    try:
        g1 = pyg_data_to_nx_digraph(g1tensor, original_graphs)
        g2_original = pyg_data_to_nx_digraph(g2tensor, noise_graphs)
        g2 = g2_original.copy()
    except ValueError as e:
        print(f"  WARNING: Could not convert graph: {e}")
        return None, None

    if len(g2.nodes()) == 0:
        print(f"  WARNING: S-graph has no nodes after conversion, skipping visualization")
        return None, None

    # Normalize node importance
    if node_importance.max() > node_importance.min():
        node_imp_norm = (node_importance - node_importance.min()) / (node_importance.max() - node_importance.min())
    else:
        node_imp_norm = node_importance

    node_list = list(g1.nodes())
    node_to_importance = {}
    for i, n in enumerate(node_list):
        node_to_importance[n] = node_imp_norm[i] if i < len(node_imp_norm) else 0

    # Translate g2
    max_x_g1 = max(data['center'][0] for _, data in g1.nodes(data=True))
    min_x_g2 = min(data['center'][0] for _, data in g2.nodes(data=True))
    translation_x = (max_x_g1 - min_x_g2) + 10.0
    for _, data in g2.nodes(data=True):
        data['center'][0] += translation_x
        if 'polygon' in data:
            poly = data['polygon']
            if isinstance(poly, Polygon):
                data['polygon'] = translate(poly, xoff=translation_x)
            else:
                data['polygon'] = Polygon([(x + translation_x, y) for x, y in poly])
        if 'limits' in data:
            data['limits'] = [[x + translation_x, y] for x, y in data['limits']]

    fig, ax = plt.subplots(figsize=(22, 12))

    # PLOT A-GRAPH (g1) - LEFT SIDE

    # Draw A-graph edges
    for u, v in g1.edges():
        ax.plot([g1.nodes[u]['center'][0], g1.nodes[v]['center'][0]],
               [g1.nodes[u]['center'][1], g1.nodes[v]['center'][1]],
               color='lightgray', linewidth=1.0, alpha=0.5, zorder=1)

    # Draw A-graph room polygons
    for n, d in g1.nodes(data=True):
        if d['type'] == 'room' and 'polygon' in d:
            poly = Polygon(d['polygon']) if not isinstance(d['polygon'], Polygon) else d['polygon']
            x, y = poly.exterior.xy
            ax.fill(x, y, alpha=0.15, fc='lightgray', ec='gray', linewidth=0.8, zorder=2)

    # Draw A-graph WS nodes
    for n, d in g1.nodes(data=True):
        if d['type'] == 'ws':
            imp = node_to_importance.get(n, 0)
            node_color = plt.cm.RdYlBu_r(imp)
            ax.scatter(d['center'][0], d['center'][1],
                      color=node_color, s=100,
                      edgecolors='black', linewidth=1.5, zorder=5)
            if 'limits' in d:
                l1, l2 = d['limits']
                ax.plot([l1[0], l2[0]], [l1[1], l2[1]], 'gray', linewidth=1.0, alpha=0.6)

    # Draw A-graph room centroids
    for n, d in g1.nodes(data=True):
        if d['type'] == 'room':
            center = d['center']
            imp = node_to_importance.get(n, 0)
            node_color = plt.cm.RdYlBu_r(imp)
            ax.scatter(center[0], center[1], color=node_color, s=120,
                      edgecolors='black', linewidth=2, zorder=5)
            ax.annotate(f'{imp:.2f}', (center[0], center[1]),
                       fontsize=7, ha='center', va='center', color='white',
                       bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))

    # PLOT S-GRAPH (g2) - RIGHT SIDE

    # Draw S-graph edges
    for u, v in g2.edges():
        ax.plot([g2.nodes[u]['center'][0], g2.nodes[v]['center'][0]],
               [g2.nodes[u]['center'][1], g2.nodes[v]['center'][1]],
               color='lightgray', linewidth=1.0, alpha=0.5, zorder=1)

    # Draw S-graph room polygons
    for n, d in g2.nodes(data=True):
        if d['type'] == 'room' and 'polygon' in d:
            poly = Polygon(d['polygon']) if not isinstance(d['polygon'], Polygon) else d['polygon']
            x, y = poly.exterior.xy
            ax.fill(x, y, alpha=0.15, fc='lightcoral', ec='gray', linewidth=0.8, zorder=2)

    # Draw S-graph WS nodes
    for n, d in g2.nodes(data=True):
        if d['type'] == 'ws':
            ax.scatter(d['center'][0], d['center'][1], color='darkgray', s=60,
                      edgecolors='gray', linewidth=1, zorder=3)
            if 'limits' in d:
                l1, l2 = d['limits']
                ax.plot([l1[0], l2[0]], [l1[1], l2[1]], 'gray', linewidth=1.0, alpha=0.6)

    # Draw S-graph room centroids
    for n, d in g2.nodes(data=True):
        if d['type'] == 'room' and str(n) != str(target_node_id):
            center = d['center']
            ax.scatter(center[0], center[1], color='darkgray', s=80,
                      edgecolors='gray', linewidth=1.5, zorder=3)

    # Draw target node on S-graph ONLY if exact match exists
    target_drawn = False
    for n, d in g2.nodes(data=True):
        if str(n) == str(target_node_id):
            center = d['center']
            ax.scatter(center[0], center[1], c='gold', s=200, marker='o',
                      edgecolors='darkorange', linewidth=2.5, zorder=10)
            ax.scatter(center[0], center[1], c='yellow', s=280, marker='o',
                      alpha=0.3, zorder=9)
            target_drawn = True
            break

    if not target_drawn:
        print(f"  WARNING: Target node '{target_node_id}' not found in S-graph!")
        print(f"  (This node may be missing from the robot's observations)")

    # MATCHING LINES (green=correct, red=wrong, orange=missing)
    if pred_perm is not None:
        for i in range(pred_perm.shape[0]):
            if i >= len(node_names1):
                continue
            if gt_perm[i].sum().item() == 0:
                continue

            row = pred_perm[i]
            if row.sum().item() == 0:
                j_gt = gt_perm[i].argmax().item()
                id1 = node_names1[i]
                if id1 not in g1.nodes or j_gt >= len(node_names2) or node_names2[j_gt] not in g2.nodes:
                    continue
                pt1 = g1.nodes[id1]['center']
                pt2 = g2.nodes[node_names2[j_gt]]['center']
                ax.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], 'orange', linestyle='--', alpha=0.7, linewidth=2)
                continue

            j = row.argmax().item()
            id1 = node_names1[i]
            if id1 not in g1.nodes or j >= len(node_names2) or node_names2[j] not in g2.nodes:
                continue

            pt1 = g1.nodes[id1]['center']
            pt2 = g2.nodes[node_names2[j]]['center']
            is_correct = (j < gt_perm.shape[1] and gt_perm[i, j] == 1)
            ax.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]],
                   color='green' if is_correct else 'red',
                   linewidth=2.5, alpha=0.8, zorder=4)

    # LEGEND
    legend_handles = [
        plt.Rectangle((0,0),1,1, facecolor='lightgray', alpha=0.5, edgecolor='gray'),
        plt.Rectangle((0,0),1,1, facecolor='lightcoral', alpha=0.5, edgecolor='gray'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gold', markersize=12, markeredgecolor='darkorange'),
        plt.Line2D([0], [0], color='green', linewidth=2.5),
        plt.Line2D([0], [0], color='red', linewidth=2.5),
        plt.Line2D([0], [0], color='orange', linewidth=2, linestyle='--'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, markeredgecolor='black'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, markeredgecolor='black'),
    ]
    legend_labels = [
        'A-graph (BIM)',
        'S-graph (Robot)',
        'Target S-node (explained)',
        'Correct match',
        'Wrong match',
        'Missing match',
        'High importance node (A-graph)',
        'Low importance node (A-graph)',
    ]
    ax.legend(legend_handles, legend_labels, loc='upper right', fontsize=9, framealpha=0.9, ncol=2)

    # Colorbar
    sm = ScalarMappable(norm=Normalize(0, 1), cmap=plt.cm.RdYlBu_r)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5)
    cbar.set_label('NODE IMPORTANCE (Red=High, Blue=Low)', fontsize=12, fontweight='bold')
    cbar.ax.tick_params(labelsize=10)

    # Title with target status
    if target_drawn:
        title_suffix = f"Target Node: {target_node_id[:50]}..."
    else:
        title_suffix = f"Target Node NOT in S-graph"

    ax.set_title(f"{title} | {title_suffix}", fontsize=14, fontweight='bold')
    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.tight_layout()
    return fig, ax

def plot_edge_importance_with_graphs(graphs_list, gt_perm, original_graphs,
                                      edge_importance, target_node_id,
                                      noise_graphs=None,
                                      title=None, save_path=None):
    """
    Edge importance visualization with both A-graph and S-graph side by side.
    """

    if noise_graphs is None:
        noise_graphs = original_graphs

    g1tensor, g2tensor = copy.deepcopy(graphs_list[0]), copy.deepcopy(graphs_list[1])

    # Check if S-graph has nodes
    if g2tensor.x.shape[0] == 0:
        print(f"  WARNING: S-graph has no nodes, skipping visualization")
        return None, None

    node_names1 = list(g1tensor.node_names)
    orig_names2 = list(g2tensor.node_names)
    perm = g2tensor.permutation.tolist()
    node_names2 = [orig_names2[p] for p in perm]

    # Convert to NetworkX
    try:
        g1 = pyg_data_to_nx_digraph(g1tensor, original_graphs)
        g2_original = pyg_data_to_nx_digraph(g2tensor, noise_graphs)
        g2 = g2_original.copy()
    except ValueError as e:
        print(f"  WARNING: Could not convert graph: {e}")
        return None, None

    if len(g2.nodes()) == 0:
        print(f"  WARNING: S-graph has no nodes after conversion, skipping visualization")
        return None, None

    # Normalize edge importance
    if edge_importance.max() > edge_importance.min():
        edge_imp_norm = (edge_importance - edge_importance.min()) / (edge_importance.max() - edge_importance.min())
    else:
        edge_imp_norm = edge_importance

    # Create edge importance mapping for A-graph (g1)
    edges_g1 = list(g1.edges())
    edge_to_importance_g1 = {}
    for idx, (u, v) in enumerate(edges_g1):
        if idx < len(edge_imp_norm):
            edge_to_importance_g1[(u, v)] = edge_imp_norm[idx]
            edge_to_importance_g1[(v, u)] = edge_imp_norm[idx]

    # Translate g2
    max_x_g1 = max(data['center'][0] for _, data in g1.nodes(data=True))
    min_x_g2 = min(data['center'][0] for _, data in g2.nodes(data=True))
    translation_x = (max_x_g1 - min_x_g2) + 10.0
    for _, data in g2.nodes(data=True):
        data['center'][0] += translation_x
        if 'polygon' in data:
            poly = data['polygon']
            if isinstance(poly, Polygon):
                data['polygon'] = translate(poly, xoff=translation_x)
            else:
                data['polygon'] = Polygon([(x + translation_x, y) for x, y in poly])
        if 'limits' in data:
            data['limits'] = [[x + translation_x, y] for x, y in data['limits']]

    fig, ax = plt.subplots(figsize=(22, 12))

    # PLOT A-GRAPH (g1) - LEFT SIDE

    # Draw A-graph edges colored by importance
    for (u, v), imp in edge_to_importance_g1.items():
        if u in g1.nodes and v in g1.nodes:
            edge_color = plt.cm.RdYlBu_r(imp)
            linewidth = 1.5 + imp * 4
            ax.plot([g1.nodes[u]['center'][0], g1.nodes[v]['center'][0]],
                   [g1.nodes[u]['center'][1], g1.nodes[v]['center'][1]],
                   color=edge_color, linewidth=linewidth, alpha=0.8, zorder=2)

    # Draw A-graph room polygons
    for n, d in g1.nodes(data=True):
        if d['type'] == 'room' and 'polygon' in d:
            poly = Polygon(d['polygon']) if not isinstance(d['polygon'], Polygon) else d['polygon']
            x, y = poly.exterior.xy
            ax.fill(x, y, alpha=0.15, fc='lightgray', ec='gray', linewidth=0.8, zorder=1)

    # Draw A-graph WS nodes
    for n, d in g1.nodes(data=True):
        if d['type'] == 'ws':
            ax.scatter(d['center'][0], d['center'][1], color='darkgray', s=60,
                      edgecolors='gray', linewidth=1, zorder=3)
            if 'limits' in d:
                l1, l2 = d['limits']
                ax.plot([l1[0], l2[0]], [l1[1], l2[1]], 'gray', linewidth=1.0, alpha=0.6)

    # Draw A-graph room centroids
    for n, d in g1.nodes(data=True):
        if d['type'] == 'room':
            center = d['center']
            ax.scatter(center[0], center[1], color='darkgray', s=80,
                      edgecolors='gray', linewidth=1.5, zorder=3)
            room_label = str(n).split('_')[-2] if '_' in str(n) else str(n)[:10]
            ax.annotate(room_label, (center[0], center[1]),
                       fontsize=7, ha='center', va='center',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    # PLOT S-GRAPH (g2) - RIGHT SIDE

    # Draw S-graph edges
    for u, v in g2.edges():
        ax.plot([g2.nodes[u]['center'][0], g2.nodes[v]['center'][0]],
               [g2.nodes[u]['center'][1], g2.nodes[v]['center'][1]],
               color='lightgray', linewidth=1.0, alpha=0.5, zorder=1)

    # Draw S-graph room polygons
    for n, d in g2.nodes(data=True):
        if d['type'] == 'room' and 'polygon' in d:
            poly = Polygon(d['polygon']) if not isinstance(d['polygon'], Polygon) else d['polygon']
            x, y = poly.exterior.xy
            ax.fill(x, y, alpha=0.15, fc='lightcoral', ec='gray', linewidth=0.8, zorder=1)

    # Draw S-graph WS nodes
    for n, d in g2.nodes(data=True):
        if d['type'] == 'ws':
            ax.scatter(d['center'][0], d['center'][1], color='darkgray', s=50,
                      edgecolors='gray', linewidth=1, zorder=3)
            if 'limits' in d:
                l1, l2 = d['limits']
                ax.plot([l1[0], l2[0]], [l1[1], l2[1]], 'gray', linewidth=1.0, alpha=0.6)

    # Draw S-graph room centroids
    for n, d in g2.nodes(data=True):
        if d['type'] == 'room' and str(n) != str(target_node_id):
            center = d['center']
            ax.scatter(center[0], center[1], color='darkgray', s=80,
                      edgecolors='gray', linewidth=1.5, zorder=3)

    # Draw target node on S-graph ONLY if exact match exists
    target_drawn = False
    for n, d in g2.nodes(data=True):
        if str(n) == str(target_node_id):
            center = d['center']
            ax.scatter(center[0], center[1], c='gold', s=200, marker='o',
                      edgecolors='darkorange', linewidth=2.5, zorder=10)
            ax.scatter(center[0], center[1], c='yellow', s=280, marker='o',
                      alpha=0.3, zorder=9)
            target_drawn = True
            break

    if not target_drawn:
        print(f"  WARNING: Target node '{target_node_id}' not found in S-graph!")
        print(f"  (This node may be missing from the robot's observations)")

    # LEGEND
    legend_handles = [
        plt.Rectangle((0,0),1,1, facecolor='lightgray', alpha=0.5, edgecolor='gray'),
        plt.Rectangle((0,0),1,1, facecolor='lightcoral', alpha=0.5, edgecolor='gray'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gold', markersize=12, markeredgecolor='darkorange'),
        plt.Line2D([0], [0], color='red', linewidth=3, alpha=0.8),
        plt.Line2D([0], [0], color='blue', linewidth=1.5, alpha=0.8),
        plt.Line2D([0], [0], color='lightgray', linewidth=1.5, alpha=0.6),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgray', markersize=8, markeredgecolor='gray'),
    ]
    legend_labels = [
        'A-graph (BIM)',
        'S-graph (Robot)',
        'Target S-node (explained)',
        'High importance edge (A-graph)',
        'Low importance edge (A-graph)',
        'S-graph edge (structural only)',
        'Node (WS or centroid)',
    ]
    ax.legend(legend_handles, legend_labels, loc='upper right', fontsize=9, framealpha=0.9, ncol=2)

    # Colorbar
    sm = ScalarMappable(norm=Normalize(0, 1), cmap=plt.cm.RdYlBu_r)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5)
    cbar.set_label('EDGE IMPORTANCE (Red=High, Blue=Low)', fontsize=12, fontweight='bold')
    cbar.ax.tick_params(labelsize=10)

    # Title with target status
    if target_drawn:
        title_suffix = f"Target Node: {target_node_id[:50]}..."
    else:
        title_suffix = f"Target Node NOT in S-graph"

    ax.set_title(f"{title} | {title_suffix}", fontsize=14, fontweight='bold')
    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.tight_layout()
    return fig, ax

def debug_missing_nodes(g1_orig, g2_orig, sample_idx=None):
    """
    Identify which S-graph nodes are missing from A-graph (by room name).
    Since A-graph (BIM) is the complete prior, S-graph should be a subset.
    """
    if sample_idx is not None:
        print(f"SAMPLE {sample_idx} - MISSING NODES DEBUG")

    # Extract room names from A-graph (BIM - complete observations)
    a_room_names = set()
    a_ws_names = set()
    for name in g1_orig.node_names:
        if "centroid" in name:
            # Extract room name (e.g., "Bedroom_7" from "..._Bedroom_7_centroid")
            parts = name.split('_')
            for i, part in enumerate(parts):
                if part in ['Livingroom', 'Bedroom', 'Bathroom', 'Kitchen', 'Balcony', 'Corridor', 'Storeroom', 'Stairs']:
                    if i + 1 < len(parts):
                        room_name = f"{part}_{parts[i+1]}"
                        a_room_names.add(room_name)
                        break
        else:
            a_ws_names.add(name)

    # Extract room names from S-graph (Robot - partial observations)
    s_room_names = set()
    s_ws_names = set()
    for name in g2_orig.node_names:
        if "centroid" in name:
            parts = name.split('_')
            for i, part in enumerate(parts):
                if part in ['Livingroom', 'Bedroom', 'Bathroom', 'Kitchen', 'Balcony', 'Corridor', 'Storeroom', 'Stairs']:
                    if i + 1 < len(parts):
                        room_name = f"{part}_{parts[i+1]}"
                        s_room_names.add(room_name)
                        break
        else:
            s_ws_names.add(name)

    # Find nodes in S-graph that are NOT in A-graph (these are errors)
    s_rooms_not_in_a = s_room_names - a_room_names
    a_rooms_not_in_s = a_room_names - s_room_names
    s_ws_not_in_a = s_ws_names - a_ws_names
    a_ws_not_in_s = a_ws_names - s_ws_names

    print(f"\nNODE COMPARISON (A-graph = BIM complete, S-graph = Robot observations)")

    print(f"\nROOM COMPARISON:")
    print(f"  A-graph rooms (BIM complete): {len(a_room_names)}")
    print(f"  S-graph rooms (Robot observed): {len(s_room_names)}")
    print(f"  Coverage: {len(s_room_names)}/{len(a_room_names)} = {len(s_room_names)/len(a_room_names)*100:.1f}%")

    if a_rooms_not_in_s:
        print(f"\n  Rooms in A-graph (BIM) but NOT observed by Robot ({len(a_rooms_not_in_s)}):")
        for room in sorted(a_rooms_not_in_s)[:10]:
            print(f"    - {room}")
        if len(a_rooms_not_in_s) > 10:
            print(f"    ... and {len(a_rooms_not_in_s) - 10} more")
        print(f"    → This is EXPECTED (robot hasn't explored these areas)")

    if s_rooms_not_in_a:
        print(f"\n  CRITICAL: Rooms in S-graph but NOT in A-graph ({len(s_rooms_not_in_a)}):")
        for room in sorted(s_rooms_not_in_a)[:10]:
            print(f"    - {room}")
        if len(s_rooms_not_in_a) > 10:
            print(f"    ... and {len(s_rooms_not_in_a) - 10} more")
        print(f"    → This is a DATA ERROR! Robot detected rooms that don't exist in BIM")

    print(f"\nWALL SEGMENT COMPARISON:")
    print(f"  A-graph WS (BIM complete): {len(a_ws_names)}")
    print(f"  S-graph WS (Robot observed): {len(s_ws_names)}")
    print(f"  Coverage: {len(s_ws_names)}/{len(a_ws_names)} = {len(s_ws_names)/len(a_ws_names)*100:.1f}%")

    if a_ws_not_in_s:
        print(f"\n  WS in A-graph but NOT observed by Robot ({len(a_ws_not_in_s)}):")
        print(f"    → This is EXPECTED (robot hasn't detected all walls)")
        if len(a_ws_not_in_s) <= 10:
            for ws in sorted(a_ws_not_in_s)[:10]:
                print(f"    - {ws[:80]}...")

    if s_ws_not_in_a:
        print(f"\n  CRITICAL: WS in S-graph but NOT in A-graph ({len(s_ws_not_in_a)}):")
        print(f"    → This is a DATA ERROR! Robot detected walls that don't exist in BIM")
        for ws in sorted(s_ws_not_in_a)[:10]:
            print(f"    - {ws[:80]}...")

    # Summary
    print("SUMMARY:")
    print(f"  Expected missing (A→S): {len(a_rooms_not_in_s)} rooms, {len(a_ws_not_in_s)} WS")
    print(f"  Data errors (S→A): {len(s_rooms_not_in_a)} rooms, {len(s_ws_not_in_a)} WS")

    if len(s_rooms_not_in_a) > 0 or len(s_ws_not_in_a) > 0:
        print(f"\n  DATA QUALITY ISSUE DETECTED")
        print(f"     Robot observed nodes that don't exist in BIM.")
        print(f"     This will cause the model to fail on these samples.")

    return {
        'a_rooms': len(a_room_names),
        's_rooms': len(s_room_names),
        'a_ws': len(a_ws_names),
        's_ws': len(s_ws_names),
        'rooms_observed_coverage': len(s_room_names) / len(a_room_names) if a_room_names else 0,
        'ws_observed_coverage': len(s_ws_names) / len(a_ws_names) if a_ws_names else 0,
        's_rooms_not_in_a': s_rooms_not_in_a,
        's_ws_not_in_a': s_ws_not_in_a,
        'a_rooms_not_in_s': a_rooms_not_in_s,
        'a_ws_not_in_s': a_ws_not_in_s
    }

### GATv2 GNN Model Node Prediction Explanations

In [ ]:
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

# Load test pairs
test_pairs = deserialize_graph_matching_dataset(DATA_PATH, "test_dataset.pkl")
train_pairs = deserialize_graph_matching_dataset(DATA_PATH, "train_dataset.pkl")

# Compute mean/std from training
mean, std = compute_mean_std(train_pairs)

# Parameters
params = ModelParams.get_default()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Path to your trained model checkpoint
CHECKPOINT_PATH = '/content/pretrained_models/best_GATv2_model.pt'

# Initialize model architecture
encoder = GATv2Encoder(params)
GATv2_model = GraphMatcher(encoder, params).to(device)

# Load trained weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
GATv2_model.load_state_dict(checkpoint['model_state_dict'])

MAX_SAMPLES_TO_ANALYZE = 100

# Identify wrong samples
wrong_samples = []
correct_samples = []

GATv2_model.eval()
with torch.no_grad():
    for idx, (g1_orig, g2_orig, P_gt) in enumerate(test_pairs[:MAX_SAMPLES_TO_ANALYZE]):
        # Find matching NetworkX graph
        g1_nx = None
        for nx_graph in original_graphs_nx:
            if len(nx_graph.nodes) == g1_orig.x.shape[0]:
                g1_nx = nx_graph
                break

        if g1_nx is None:
            continue

        # Get predictions
        hard_assign, pred_labels, true_labels, _ = get_misclassified_node_and_predictions(
            GATv2_model, g1_orig, g2_orig, P_gt, device, mean, std
        )

        # Check prediction quality
        correct_count = 0
        total_matches = 0
        for j in range(P_gt.shape[0]):
            if P_gt[j].sum().item() > 0:
                total_matches += 1
                if pred_labels[j].item() == true_labels[j].item():
                    correct_count += 1

        is_perfect = (correct_count == total_matches)

        sample_info = {
            'idx': idx,
            'g1_orig': g1_orig,
            'g2_orig': g2_orig,
            'P_gt': P_gt,
            'g1_nx': g1_nx,
            'n1': g1_orig.x.shape[0],
            'n2': g2_orig.x.shape[0],
            'correct_count': correct_count,
            'total_matches': total_matches,
            'accuracy': correct_count / total_matches if total_matches > 0 else 0
        }

        if not is_perfect:
            wrong_samples.append(sample_info)
        else:
            correct_samples.append(sample_info)

print(f"\nSummary: {len(wrong_samples)} wrong, {len(correct_samples)} correct out of {len(test_pairs[:MAX_SAMPLES_TO_ANALYZE])} samples")
print(f"Correct percentage: {len(correct_samples)/len(test_pairs[:MAX_SAMPLES_TO_ANALYZE])*100:.1f}%")

#### Wrong Prediction Analysis

In [ ]:
# Generate explanations for wrong samples
print("Generating Explanations for WRONG Predictions")

num_wrong_to_explain = 20

# In your main loop where you call the explainer
for i, sample in enumerate(wrong_samples[:num_wrong_to_explain]):
    print(f"WRONG PREDICTION #{i+1} (Original Sample {sample['idx']+1})")
    print(f"Overall Sample Accuracy: {sample['accuracy']:.1%} ({sample['correct_count']}/{sample['total_matches']} correct matches)")

    # Get explanations with feature importance
    node_importance, edge_importance, feature_importance, node_idx, pred_class, true_class, accuracy, hard_assign = get_gnnexplainer_importance_for_misclassified(
        GATv2_model, sample['g1_orig'], sample['g2_orig'], sample['P_gt'].cpu(), device, mean, std, epochs=100
    )

    # Print feature importance summary
    if feature_importance is not None:
        feature_names = ['Type_Room', 'Type_WS', 'Centroid_X', 'Centroid_Y', 'Normal_X', 'Normal_Y', 'Segment_Length']
        print(f"\n  FEATURE IMPORTANCE SUMMARY for node {node_idx}:")
        sorted_idx = np.argsort(feature_importance)[::-1]
        for rank, idx in enumerate(sorted_idx[:3]):
            print(f"    #{rank+1}: {feature_names[idx]} = {feature_importance[idx]:.4f}")
    # Get S-graph node names in permuted order
    permuted_indices = sample['g2_orig'].permutation.tolist() if hasattr(sample['g2_orig'], 'permutation') else list(range(sample['g2_orig'].x.shape[0]))
    s_graph_nodes_permuted = [sample['g2_orig'].node_names[i] for i in permuted_indices]

    # Target node is the S-node that was PREDICTED (where the gold dot goes)
    if pred_class < len(s_graph_nodes_permuted):
        target_node_id = s_graph_nodes_permuted[pred_class]
    else:
        target_node_id = s_graph_nodes_permuted[0]

    true_node_id = s_graph_nodes_permuted[true_class] if true_class < len(s_graph_nodes_permuted) else "N/A"

    print(f"  Explaining A-node {node_idx}")
    print(f"  Predicted S-node (index {pred_class}): {str(target_node_id)[:60]}...")
    print(f"  True S-node (index {true_class}): {str(true_node_id)[:60]}...")
    print(f"  Classifier Accuracy: {accuracy:.3f}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # Plot 1: Node importance (with matching lines)
    fig1, ax1 = plot_node_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        node_importance=node_importance,
        target_node_id=target_node_id,
        pred_perm=hard_assign,
        noise_graphs=noise_graphs_nx,
        title=f"WRONG | Sample {sample['idx']+1} | NODE IMPORTANCE",
        save_path=None
    )
    if fig1 is not None:
        plt.show()
        plt.close(fig1)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # Plot 2: Edge importance (no matching lines)
    fig2, ax2 = plot_edge_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        edge_importance=edge_importance,
        target_node_id=target_node_id,
        noise_graphs=noise_graphs_nx,
        title=f"WRONG | Sample {sample['idx']+1} | EDGE IMPORTANCE",
        save_path=None
    )
    if fig2 is not None:
        plt.show()
        plt.close(fig2)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

#### Correct Prediction Analysis

In [ ]:
# Generate explanations for correct samples
print("Generating Explanations for CORRECT Predictions")

num_correct_to_explain = 20

for i, sample in enumerate(correct_samples[:num_correct_to_explain]):
    print(f"CORRECT PREDICTION #{i+1} (Original Sample {sample['idx']+1})")
    print(f"Overall Sample Accuracy: {sample['accuracy']:.1%} ({sample['correct_count']}/{sample['total_matches']} correct matches)")

    # Get explanations with feature importance
    node_importance, edge_importance, feature_importance, node_idx, pred_class, true_class, accuracy, hard_assign = get_gnnexplainer_importance_for_correct(
        GATv2_model, sample['g1_orig'], sample['g2_orig'], sample['P_gt'].cpu(), device, mean, std, epochs=100
    )

    # Get S-graph node names
    permuted_indices = sample['g2_orig'].permutation.tolist() if hasattr(sample['g2_orig'], 'permutation') else list(range(sample['g2_orig'].x.shape[0]))
    s_graph_nodes_permuted = [sample['g2_orig'].node_names[i] for i in permuted_indices]

    if pred_class < len(s_graph_nodes_permuted):
        target_node_id = s_graph_nodes_permuted[pred_class]
    else:
        target_node_id = s_graph_nodes_permuted[0]

    print(f"  Explaining A-node {node_idx}")
    print(f"  Correctly predicted S-node (index {pred_class}): {str(target_node_id)[:60]}...")
    print(f"  Classifier Accuracy: {accuracy:.3f}")

    # Print feature importance summary for the target node
    if feature_importance is not None:
        feature_names = ['Type_Room', 'Type_WS', 'Centroid_X', 'Centroid_Y', 'Normal_X', 'Normal_Y', 'Segment_Length']
        print(f"\n  FEATURE IMPORTANCE SUMMARY for node {node_idx}:")
        sorted_idx = np.argsort(feature_importance)[::-1]
        for rank, idx in enumerate(sorted_idx[:3]):
            print(f"    #{rank+1}: {feature_names[idx]} = {feature_importance[idx]:.4f}")
        print(f"    Least important: {feature_names[sorted_idx[-1]]} = {feature_importance[sorted_idx[-1]]:.4f}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # PLOT 1: Node Importance
    fig1, ax1 = plot_node_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        node_importance=node_importance,
        target_node_id=target_node_id,
        pred_perm=hard_assign,
        noise_graphs=noise_graphs_nx,
        title=f"CORRECT | Sample {sample['idx']+1} | NODE IMPORTANCE | S-node: {pred_class}",
        save_path=None
    )
    if fig1 is not None:
        plt.show()
        plt.close(fig1)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # PLOT 2: Edge Importance
    fig2, ax2 = plot_edge_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        edge_importance=edge_importance,
        target_node_id=target_node_id,
        noise_graphs=noise_graphs_nx,
        title=f"CORRECT | Sample {sample['idx']+1} | EDGE IMPORTANCE | S-node: {pred_class}",
        save_path=None
    )
    if fig2 is not None:
        plt.show()
        plt.close(fig2)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

#### Inspect Specific Sample

In [ ]:
'''
Problem: Target Node: 8754_02462b3dc06fbbf089a6df1c689f1a9b_Bedroom_6_ws_11... not found for graph visualization

- Predicted S-node (index 0): 8754_02462b3dc06fbbf089a6df1c689f1a9b_Bedroom_6_ws_11...
- True S-node (index 3): 8754_02462b3dc06fbbf089a6df1c689f1a9b_Balcony_3_centroid...

Our graph visualization function has to match (red line) a WS node (Bedroom_6_ws_11) in test_dataset.pkl
 - the copy of WS node to match doesn't exist in the S-graph data of noise.pkl
 - by default, we set the function to match to print a debug statement

Our Theory: The problem is incomplete S-graph data in noise.pkl for graph visualization
'''

# Selected test sample
sample_idx = 7
g1_orig, g2_orig, P_gt = test_pairs[sample_idx]

# Check if it's in wrong_samples
is_in_wrong = any(s['idx'] == sample_idx for s in wrong_samples)
is_in_correct = any(s['idx'] == sample_idx for s in correct_samples)

print(f"\nSample {sample_idx}:")
print(f"  In wrong_samples: {is_in_wrong}")
print(f"  In correct_samples: {is_in_correct}")

if is_in_wrong:
    # Find the sample info
    sample_info = next(s for s in wrong_samples if s['idx'] == sample_idx)
    print(f"  Status: WRONG")
    print(f"  Accuracy: {sample_info['accuracy']:.1%}")
    print(f"  Matches: {sample_info['correct_count']}/{sample_info['total_matches']}")
else:
    print(f"  Status: CORRECT or not in samples list")

# Run debug functions
debug_missing_nodes(g1_orig, g2_orig, sample_idx=sample_idx)

### GraphSAGE GNN Model Node Prediction Explanations

In [ ]:
ORIGINAL_PATH = os.path.join(DATA_PATH, "original.pkl")
NOISE_PATH = os.path.join(DATA_PATH, "noise.pkl")

# Load test pairs
test_pairs = deserialize_graph_matching_dataset(DATA_PATH, "test_dataset.pkl")
train_pairs = deserialize_graph_matching_dataset(DATA_PATH, "train_dataset.pkl")

# Compute mean/std from training
mean, std = compute_mean_std(train_pairs)

# Parameters
params = ModelParams.get_default()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Path to your trained model checkpoint
CHECKPOINT_PATH = '/content/pretrained_models/best_GraphSAGE_model.pt'

# Initialize model architecture
encoder = GraphSAGEEncoder(params)
graphSAGE_model = GraphMatcher(encoder, params).to(device)

# Load trained weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
graphSAGE_model.load_state_dict(checkpoint['model_state_dict'])

MAX_SAMPLES_TO_ANALYZE = 100

# Identify wrong samples
wrong_samples = []
correct_samples = []

graphSAGE_model.eval()
with torch.no_grad():
    for idx, (g1_orig, g2_orig, P_gt) in enumerate(test_pairs[:MAX_SAMPLES_TO_ANALYZE]):
        # Find matching NetworkX graph
        g1_nx = None
        for nx_graph in original_graphs_nx:
            if len(nx_graph.nodes) == g1_orig.x.shape[0]:
                g1_nx = nx_graph
                break

        if g1_nx is None:
            continue

        # Get predictions
        hard_assign, pred_labels, true_labels, _ = get_misclassified_node_and_predictions(
            graphSAGE_model, g1_orig, g2_orig, P_gt, device, mean, std
        )

        # Check prediction quality
        correct_count = 0
        total_matches = 0
        for j in range(P_gt.shape[0]):
            if P_gt[j].sum().item() > 0:
                total_matches += 1
                if pred_labels[j].item() == true_labels[j].item():
                    correct_count += 1

        is_perfect = (correct_count == total_matches)

        sample_info = {
            'idx': idx,
            'g1_orig': g1_orig,
            'g2_orig': g2_orig,
            'P_gt': P_gt,
            'g1_nx': g1_nx,
            'n1': g1_orig.x.shape[0],
            'n2': g2_orig.x.shape[0],
            'correct_count': correct_count,
            'total_matches': total_matches,
            'accuracy': correct_count / total_matches if total_matches > 0 else 0
        }

        if not is_perfect:
            wrong_samples.append(sample_info)
        else:
            correct_samples.append(sample_info)

print(f"\nSummary: {len(wrong_samples)} wrong, {len(correct_samples)} correct out of {len(test_pairs[:MAX_SAMPLES_TO_ANALYZE])} samples")
print(f"Correct percentage: {len(correct_samples)/len(test_pairs[:MAX_SAMPLES_TO_ANALYZE])*100:.1f}%")

#### Wrong Prediction Analysis

In [ ]:
# Generate explanations for wrong samples
print("Generating Explanations for WRONG Predictions")

num_wrong_to_explain = 20

# In your main loop where you call the explainer
for i, sample in enumerate(wrong_samples[:num_wrong_to_explain]):
    print(f"WRONG PREDICTION #{i+1} (Original Sample {sample['idx']+1})")
    print(f"Overall Sample Accuracy: {sample['accuracy']:.1%} ({sample['correct_count']}/{sample['total_matches']} correct matches)")

    # Get explanations with feature importance
    node_importance, edge_importance, feature_importance, node_idx, pred_class, true_class, accuracy, hard_assign = get_gnnexplainer_importance_for_misclassified(
        GATv2_model, sample['g1_orig'], sample['g2_orig'], sample['P_gt'].cpu(), device, mean, std, epochs=100
    )

    # Print feature importance summary
    if feature_importance is not None:
        feature_names = ['Type_Room', 'Type_WS', 'Centroid_X', 'Centroid_Y', 'Normal_X', 'Normal_Y', 'Segment_Length']
        print(f"\n  FEATURE IMPORTANCE SUMMARY for node {node_idx}:")
        sorted_idx = np.argsort(feature_importance)[::-1]
        for rank, idx in enumerate(sorted_idx[:3]):
            print(f"    #{rank+1}: {feature_names[idx]} = {feature_importance[idx]:.4f}")
    # Get S-graph node names in permuted order
    permuted_indices = sample['g2_orig'].permutation.tolist() if hasattr(sample['g2_orig'], 'permutation') else list(range(sample['g2_orig'].x.shape[0]))
    s_graph_nodes_permuted = [sample['g2_orig'].node_names[i] for i in permuted_indices]

    # Target node is the S-node that was PREDICTED (where the gold dot goes)
    if pred_class < len(s_graph_nodes_permuted):
        target_node_id = s_graph_nodes_permuted[pred_class]
    else:
        target_node_id = s_graph_nodes_permuted[0]

    true_node_id = s_graph_nodes_permuted[true_class] if true_class < len(s_graph_nodes_permuted) else "N/A"

    print(f"  Explaining A-node {node_idx}")
    print(f"  Predicted S-node (index {pred_class}): {str(target_node_id)[:60]}...")
    print(f"  True S-node (index {true_class}): {str(true_node_id)[:60]}...")
    print(f"  Classifier Accuracy: {accuracy:.3f}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # Plot 1: Node importance (with matching lines)
    fig1, ax1 = plot_node_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        node_importance=node_importance,
        target_node_id=target_node_id,
        pred_perm=hard_assign,
        noise_graphs=noise_graphs_nx,
        title=f"WRONG | Sample {sample['idx']+1} | NODE IMPORTANCE",
        save_path=None
    )
    if fig1 is not None:
        plt.show()
        plt.close(fig1)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # Plot 2: Edge importance (no matching lines)
    fig2, ax2 = plot_edge_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        edge_importance=edge_importance,
        target_node_id=target_node_id,
        noise_graphs=noise_graphs_nx,
        title=f"WRONG | Sample {sample['idx']+1} | EDGE IMPORTANCE",
        save_path=None
    )
    if fig2 is not None:
        plt.show()
        plt.close(fig2)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

#### Correct Prediction Analysis

In [ ]:
# Generate explanations for correct samples
print("Generating Explanations for CORRECT Predictions")

num_correct_to_explain = 20

for i, sample in enumerate(correct_samples[:num_correct_to_explain]):
    print(f"CORRECT PREDICTION #{i+1} (Original Sample {sample['idx']+1})")
    print(f"Overall Sample Accuracy: {sample['accuracy']:.1%} ({sample['correct_count']}/{sample['total_matches']} correct matches)")

    # Get explanations with feature importance
    node_importance, edge_importance, feature_importance, node_idx, pred_class, true_class, accuracy, hard_assign = get_gnnexplainer_importance_for_correct(
        graphSAGE_model, sample['g1_orig'], sample['g2_orig'], sample['P_gt'].cpu(), device, mean, std, epochs=100
    )

    # Get S-graph node names
    permuted_indices = sample['g2_orig'].permutation.tolist() if hasattr(sample['g2_orig'], 'permutation') else list(range(sample['g2_orig'].x.shape[0]))
    s_graph_nodes_permuted = [sample['g2_orig'].node_names[i] for i in permuted_indices]

    if pred_class < len(s_graph_nodes_permuted):
        target_node_id = s_graph_nodes_permuted[pred_class]
    else:
        target_node_id = s_graph_nodes_permuted[0]

    print(f"  Explaining A-node {node_idx}")
    print(f"  Correctly predicted S-node (index {pred_class}): {str(target_node_id)[:60]}...")
    print(f"  Classifier Accuracy: {accuracy:.3f}")

    # Print feature importance summary for the target node
    if feature_importance is not None:
        feature_names = ['Type_Room', 'Type_WS', 'Centroid_X', 'Centroid_Y', 'Normal_X', 'Normal_Y', 'Segment_Length']
        print(f"\n  FEATURE IMPORTANCE SUMMARY for node {node_idx}:")
        sorted_idx = np.argsort(feature_importance)[::-1]
        for rank, idx in enumerate(sorted_idx[:3]):
            print(f"    #{rank+1}: {feature_names[idx]} = {feature_importance[idx]:.4f}")
        print(f"    Least important: {feature_names[sorted_idx[-1]]} = {feature_importance[sorted_idx[-1]]:.4f}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # PLOT 1: Node Importance
    fig1, ax1 = plot_node_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        node_importance=node_importance,
        target_node_id=target_node_id,
        pred_perm=hard_assign,
        noise_graphs=noise_graphs_nx,
        title=f"CORRECT | Sample {sample['idx']+1} | NODE IMPORTANCE | S-node: {pred_class}",
        save_path=None
    )
    if fig1 is not None:
        plt.show()
        plt.close(fig1)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

    # Load graphs
    with open(ORIGINAL_PATH, 'rb') as f:
        original_graphs_nx = pickle.load(f)
    with open(NOISE_PATH, 'rb') as f:
        noise_graphs_nx = pickle.load(f)

    # PLOT 2: Edge Importance
    fig2, ax2 = plot_edge_importance_with_graphs(
        graphs_list=[sample['g1_orig'], sample['g2_orig']],
        gt_perm=sample['P_gt'].cpu(),
        original_graphs=original_graphs_nx,
        edge_importance=edge_importance,
        target_node_id=target_node_id,
        noise_graphs=noise_graphs_nx,
        title=f"CORRECT | Sample {sample['idx']+1} | EDGE IMPORTANCE | S-node: {pred_class}",
        save_path=None
    )
    if fig2 is not None:
        plt.show()
        plt.close(fig2)
    else:
        print(f"  Skipping visualization for sample {sample['idx']+1}")

#### Inspect Specific Sample

In [ ]:
# Selected test sample
sample_idx = 2
g1_orig, g2_orig, P_gt = test_pairs[sample_idx]

# Check if it's in wrong_samples
is_in_wrong = any(s['idx'] == sample_idx for s in wrong_samples)
is_in_correct = any(s['idx'] == sample_idx for s in correct_samples)

print(f"\nSample {sample_idx}:")
print(f"  In wrong_samples: {is_in_wrong}")
print(f"  In correct_samples: {is_in_correct}")

if is_in_wrong:
    # Find the sample info
    sample_info = next(s for s in wrong_samples if s['idx'] == sample_idx)
    print(f"  Status: WRONG")
    print(f"  Accuracy: {sample_info['accuracy']:.1%}")
    print(f"  Matches: {sample_info['correct_count']}/{sample_info['total_matches']}")
else:
    print(f"  Status: CORRECT or not in samples list")

# Run debug functions
debug_missing_nodes(g1_orig, g2_orig, sample_idx=sample_idx)

## Automated Hyperparameter Optimization using Optuna

In [ ]:
def create_smaller_datasets(train_dataset, val_dataset, test_dataset, sample_ratio=0.5):
    """Create smaller datasets for faster hyperparameter optimization."""

    train_size = int(len(train_dataset) * sample_ratio)
    val_size = int(len(val_dataset) * sample_ratio)
    test_size = int(len(test_dataset) * sample_ratio)

    torch.manual_seed(42)
    train_indices = torch.randperm(len(train_dataset))[:train_size]
    val_indices = torch.randperm(len(val_dataset))[:val_size]
    test_indices = torch.randperm(len(test_dataset))[:test_size]

    small_train_dataset = Subset(train_dataset, train_indices)
    small_val_dataset = Subset(val_dataset, val_indices)
    small_test_dataset = Subset(test_dataset, test_indices)

    print(f"\nCreated smaller datasets for optimization:")
    print(f"  Train: {len(small_train_dataset)} samples (original: {len(train_dataset)})")
    print(f"  Val: {len(small_val_dataset)} samples (original: {len(val_dataset)})")
    print(f"  Test: {len(small_test_dataset)} samples (original: {len(test_dataset)})")

    return small_train_dataset, small_val_dataset, small_test_dataset


def train_epoch(model, train_loader, optimizer, device):
    """Train for one epoch using proper batch handling."""
    model.train()
    total_loss = 0
    num_batches = 0

    for batch1, batch2, perm_list in train_loader:
        # Move entire batch to device
        batch1 = batch1.to(device)
        batch2 = batch2.to(device)
        perm_list = [p.to(device) for p in perm_list]

        # Pass entire batches to model (returns list of S_pred for each graph in batch)
        S_pred_list, _ = model(batch1, batch2)

        # Compute loss for each graph pair in the batch
        batch_loss = 0
        for i, S_pred in enumerate(S_pred_list):
            P_gt = perm_list[i]
            loss = permutation_loss(S_pred, P_gt)
            batch_loss += loss

        # Average loss over batch
        batch_loss = batch_loss / len(S_pred_list)

        # Backward pass
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    return avg_loss


def create_model_with_params(trial, encoder_type='GATv2', use_small_sinkhorn=True):
    """Create model with hyperparameters from Optuna trial."""

    hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128, 256])
    output_dim = trial.suggest_categorical('output_dim', [16, 32, 64, 128])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    num_heads = trial.suggest_categorical('num_heads', [2, 4, 8]) if encoder_type in ['GATv2'] else 4
    dropout = trial.suggest_float('dropout', 0.0, 0.3)
    attn_dropout = trial.suggest_float('attn_dropout', 0.0, 0.3) if encoder_type in ['GATv2'] else dropout
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)

    if use_small_sinkhorn:
        sinkhorn_iterations = trial.suggest_int('sinkhorn_iterations', 5, 20)
    else:
        sinkhorn_iterations = trial.suggest_int('sinkhorn_iterations', 10, 50)

    params = ModelParams(
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        num_layers=num_layers,
        num_heads=num_heads,
        dropout=dropout,
        attn_dropout=attn_dropout,
        sinkhorn_iterations=sinkhorn_iterations,
        learning_rate=lr,
        weight_decay=weight_decay
    )

    if encoder_type == 'GATv2':
        encoder = GATv2Encoder(params)
    elif encoder_type == 'GCN':
        encoder = GCNEncoder(params)
    elif encoder_type == 'GraphSAGE':
        encoder = GraphSAGEEncoder(params)
    elif encoder_type == 'GIN':
        encoder = GINEncoder(params)
    elif encoder_type == 'GraphTransformer':
        encoder = GraphTransformerEncoder(params)
    else:
        raise ValueError(f"Unknown encoder type: {encoder_type}")

    model = GraphMatcher(encoder, params)

    return model, lr, weight_decay, params


# Objective function optimizes LOSS
def objective(trial, train_loader, val_loader, device, encoder_type='GATv2', n_epochs=10):
    """
    Objective function for Optuna optimization.
    """

    model, lr, weight_decay, params = create_model_with_params(trial, encoder_type, use_small_sinkhorn=True)
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Track best validation loss
    best_val_loss = float('inf')

    for epoch in range(n_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, device)
        val_metrics, _ = evaluate(model, val_loader, device, verbose=False)
        val_loss = val_metrics.get('loss', 0)

        # Update best validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Report validation loss for pruning
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_loss


# EncoderOptimizer with loss-based optimization
class EncoderOptimizer:
    """Manages hyperparameter optimization for multiple encoders."""

    def __init__(self, train_loader, val_loader, test_loader, device):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.results = {}
        self.best_models = {}
        self.best_params = {}

    def optimize_encoder(self, encoder_type, n_trials=10, n_epochs_per_trial=10):
        """Optimize a single encoder."""

        print(f"OPTIMIZING {encoder_type} ENCODER")
        print(f"Trials: {n_trials}, Epochs per trial: {n_epochs_per_trial}")

        # direction='minimize' to optimize for lowest loss
        study_name = f"{encoder_type}_optimization_loss"
        study = optuna.create_study(
            direction='minimize',
            study_name=study_name,
            pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
            load_if_exists=True
        )

        def objective_with_encoder(trial):
            return objective(trial, self.train_loader, self.val_loader,
                           self.device, encoder_type, n_epochs_per_trial)

        start_time = time.time()
        study.optimize(objective_with_encoder, n_trials=n_trials, show_progress_bar=True)
        elapsed_time = time.time() - start_time

        # Store results
        self.results[encoder_type] = {
            'best_value': study.best_value,
            'best_loss': study.best_value,
            'best_params': study.best_trial.params,
            'n_trials': n_trials,
            'elapsed_time': elapsed_time,
            'study': study
        }

        print(f"\n{encoder_type} optimization complete!")
        print(f"   Best validation loss: {self.results[encoder_type]['best_loss']:.6f}")
        print(f"   Time: {elapsed_time/60:.2f} minutes")

        return study

    def optimize_all_encoders(self, encoder_types, trials_per_encoder=10, epochs_per_trial=10):
        """Optimize all encoders automatically."""

        print("AUTOMATED ENCODER OPTIMIZATION")
        print(f"Encoders to optimize: {encoder_types}")
        print(f"Trials per encoder: {trials_per_encoder}")
        print(f"Epochs per trial: {epochs_per_trial}")

        for encoder_type in encoder_types:
            self.optimize_encoder(encoder_type, trials_per_encoder, epochs_per_trial)

        self.print_optimization_summary()

        return self.results

    def print_optimization_summary(self):
        """Print summary of all optimization results."""

        print("OPTIMIZATION SUMMARY")
        print(f"{'Encoder':<12} {'Best Loss':<12} {'Trials':<8} {'Time (min)':<12}")

        for encoder, results in self.results.items():
            print(f"{encoder:<12} {results['best_loss']:.6f}   {results['n_trials']:<8} {results['elapsed_time']/60:.2f}")

        # Find best overall (lowest loss)
        best_encoder = min(self.results.keys(), key=lambda x: self.results[x]['best_loss'])
        print(f"BEST OVERALL MODEL (Lowest Loss): {best_encoder}")
        print(f"   Best validation loss: {self.results[best_encoder]['best_loss']:.6f}")

    def train_final_models(self, n_epochs=20):
        """Train final models for each encoder using best parameters."""

        print("TRAINING FINAL MODELS WITH BEST PARAMETERS")

        for encoder, results in self.results.items():
            print(f"Training final {encoder} model...")

            best_params = results['best_params']

            model, test_metrics, train_losses, val_f1_scores = self.train_single_final_model(
                encoder, best_params, n_epochs
            )

            self.best_models[encoder] = model
            self.results[encoder]['final_test_f1'] = test_metrics['f1']
            self.results[encoder]['final_test_precision'] = test_metrics['precision']
            self.results[encoder]['final_test_recall'] = test_metrics['recall']
            self.results[encoder]['final_test_accuracy'] = test_metrics['accuracy']
            self.results[encoder]['final_train_losses'] = train_losses
            self.results[encoder]['final_val_f1_scores'] = val_f1_scores

            print(f"\n  {encoder} Final Test Results:")
            print(f"    F1: {test_metrics['f1']:.4f}")
            print(f"    Precision: {test_metrics['precision']:.4f}")
            print(f"    Recall: {test_metrics['recall']:.4f}")

        self.print_final_comparison()

        return self.best_models

    def train_single_final_model(self, encoder_type, best_params, n_epochs=20):
        """Train a single final model with best parameters."""

        params = ModelParams(
            hidden_dim=best_params.get('hidden_dim', 128),
            output_dim=best_params.get('output_dim', 32),
            num_layers=best_params.get('num_layers', 2),
            num_heads=best_params.get('num_heads', 4),
            dropout=best_params.get('dropout', 0.1),
            attn_dropout=best_params.get('attn_dropout', 0.1),
            sinkhorn_iterations=best_params.get('sinkhorn_iterations', 30),
            learning_rate=best_params.get('lr', 0.001),
            weight_decay=best_params.get('weight_decay', 1e-5),
            num_epochs=n_epochs,
            patience=20
        )

        if encoder_type == 'GATv2':
            encoder = GATv2Encoder(params)
        elif encoder_type == 'GCN':
            encoder = GCNEncoder(params)
        elif encoder_type == 'GraphSAGE':
            encoder = GraphSAGEEncoder(params)
        elif encoder_type == 'GIN':
            encoder = GINEncoder(params)
        elif encoder_type == 'GraphTransformer':
            encoder = GraphTransformerEncoder(params)
        else:
            raise ValueError(f"Unknown encoder type: {encoder_type}")

        model = GraphMatcher(encoder, params).to(self.device)

        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=params.learning_rate,
                                      weight_decay=params.weight_decay)

        best_val_loss = float('inf')
        best_val_f1 = 0.0
        best_model_state = None
        patience_counter = 0
        train_losses = []
        val_f1_scores = []

        for epoch in range(n_epochs):
            train_loss = train_epoch(model, self.train_loader, optimizer, self.device)
            train_losses.append(train_loss)

            val_metrics, _ = evaluate(model, self.val_loader, self.device)
            val_f1 = val_metrics['f1']
            val_loss = val_metrics.get('loss', 0)
            val_f1_scores.append(val_f1)

            if (epoch + 1) % 1 == 0:
                print(f"  Epoch {epoch+1:3d}/{n_epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")

            # Early stopping based on loss
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                if patience_counter >= params.patience:
                    print(f"  Early stopping at epoch {epoch+1} (Best validation loss: {best_val_loss:.4f})")
                    break

        model.load_state_dict(best_model_state)
        test_metrics, _ = evaluate(model, self.test_loader, self.device)

        return model, test_metrics, train_losses, val_f1_scores

    def print_final_comparison(self):
        """Print final comparison of all models."""

        print("FINAL MODEL COMPARISON")
        print(f"{'Encoder':<12} {'Test F1':<10} {'Precision':<10} {'Recall':<10} {'Accuracy':<10} {'Params':<12}")

        for encoder, results in self.results.items():
            params_count = sum(p.numel() for p in self.best_models[encoder].parameters())
            print(f"{encoder:<12} {results['final_test_f1']:.4f}     {results['final_test_precision']:.4f}     "
                  f"{results['final_test_recall']:.4f}     {results['final_test_accuracy']:.4f}     {params_count:,}")

        best_encoder = max(self.results.keys(), key=lambda x: self.results[x]['final_test_f1'])
        print(f"BEST OVERALL MODEL (Highest Test F1): {best_encoder}")
        print(f"   Test F1: {self.results[best_encoder]['final_test_f1']:.4f}")

    def plot_comparison(self):
        """Plot comparison of all models."""
        import matplotlib.pyplot as plt

        encoders = list(self.results.keys())
        test_f1 = [self.results[encoder]['final_test_f1'] for encoder in encoders]
        test_precision = [self.results[encoder]['final_test_precision'] for encoder in encoders]
        test_recall = [self.results[encoder]['final_test_recall'] for encoder in encoders]

        x = np.arange(len(encoders))
        width = 0.25

        fig, ax = plt.subplots(figsize=(12, 6))
        bars1 = ax.bar(x - width, test_f1, width, label='F1 Score', color='green')
        bars2 = ax.bar(x, test_precision, width, label='Precision', color='blue')
        bars3 = ax.bar(x + width, test_recall, width, label='Recall', color='orange')

        ax.set_xlabel('Model')
        ax.set_ylabel('Score')
        ax.set_title('Model Comparison on Test Set')
        ax.set_xticks(x)
        ax.set_xticklabels(encoders)
        ax.legend()
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


print("AUTOMATED MODEL OPTIMIZATION AND COMPARISON")

# Create smaller datasets for faster optimization
small_train_dataset, small_val_dataset, small_test_dataset = create_smaller_datasets(
    train_dataset, val_dataset, test_dataset, sample_ratio=0.1)

# Create dataloaders
batch_size_opt = 16
small_train_loader = DataLoader(small_train_dataset, batch_size=batch_size_opt,
                                shuffle=True, collate_fn=collate_pyg_matching)
small_val_loader = DataLoader(small_val_dataset, batch_size=batch_size_opt,
                              shuffle=False, collate_fn=collate_pyg_matching)
small_test_loader = DataLoader(small_test_dataset, batch_size=batch_size_opt,
                               shuffle=False, collate_fn=collate_pyg_matching)

# Create optimizer
optimizer = EncoderOptimizer(small_train_loader, small_val_loader, small_test_loader, device)

# Define encoders to test
encoder_types = ['GATv2', 'GCN', 'GraphSAGE', 'GIN', 'GraphTransformer']

# Run optimization for all encoders
optimizer.optimize_all_encoders(
    encoder_types,
    trials_per_encoder=10,
    epochs_per_trial=10
)

# Train final models with best parameters
optimizer.train_final_models(n_epochs=20)

# Plot comparison
optimizer.plot_comparison()

# Save all results
results_df = pd.DataFrame([
    {
        'Encoder': encoder,
        'Optimization_Best_Loss': results['best_loss'],
        'Test_F1': results['final_test_f1'],
        'Test_Precision': results['final_test_precision'],
        'Test_Recall': results['final_test_recall'],
        'Test_Accuracy': results['final_test_accuracy'],
        'Best_Params': str(results['best_params'])
    }
    for encoder, results in optimizer.results.items()
])

results_df.to_csv('/content/results/encoder_loss_optimization_results.csv', index=False)